# 03 · Transfer status for the edge cases and a 2,000-event uniform sample

`GET https://li.quest/v1/status?txHash=<hash>` for the edge cases the data profile found and for 2,000 further events drawn uniformly at random, excluding the 600 already sampled in `01_reference_data.ipynb` §5. Every fetch cell reads its snapshot under `data/reference/` when the file exists and calls the network only when it does not. The raw files are not in this repository, so the committed outputs are the record of the run. The LI.FI key is read from `LIFI_API_KEY` in the environment, else from the gitignored `.env`, sent as the `x-lifi-api-key` header, and never printed or written. Sections are numbered.

Snapshots written here: `lifi_api_transfer_status_edge_cases.csv` and `_raw.jsonl` (§4), `lifi_api_transfer_status_sample_2600.csv` and `_raw.jsonl` (§6). USD figures labelled "provided" are the profile's first USD column (`00_explore.ipynb` §11: `minAmount / 10^decimals × price_usd` from `prices.csv`, no fallback); USD figures from the API (`amountUSD`, `feeCosts[].amountUSD`) are LI.FI's own pricing and depend on reference data.

## 1. Setup: paths, key, HTTP helper with rate-limit pacing, snapshot helpers, flattening

In [1]:
import json
import math
import os
import time
import datetime as dt
from collections import defaultdict, Counter
from decimal import Decimal
from pathlib import Path

import pandas as pd
import requests

from lifi import paths
from lifi.load import load_transfers, load_prices

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 150)
pd.set_option("display.max_colwidth", 70)

ROOT = paths.REPO_ROOT                 # the repository root, where the gitignored .env lives
DATA = paths.data_dir()                # LIFI_DATA_DIR, else data/ in the repository; only the two CSVs are read from it, by lifi.load too
DATA_LABEL = "LIFI_DATA_DIR" if os.environ.get("LIFI_DATA_DIR") else "data under the repository root"   # printed instead of a path
REF = paths.reference_dir()            # data/reference in the repository, whatever LIFI_DATA_DIR says: snapshots are read from it and, when missing, fetched into it
MANIFEST = REF / "snapshot_manifest.json"
LIFI = "https://li.quest/v1"
STATUS_URL = f"{LIFI}/status?txHash=<tx_hash>"

ZERO_ADDRESS = "0x0000000000000000000000000000000000000000"
SENTINEL_RECEIVER = "0x11f111f111f111f111f111f111f111f111f111f1"
SOURCE_CHAIN_ID = {"ethereum": 1, "optimism": 10, "bnb": 56, "polygon": 137, "base": 8453, "arbitrum": 42161}   # 01 §2
SEED = 20261029                       # every random choice in this notebook
PACE_SECONDS = 0.65                   # 100 requests per minute with the key; ~92 per minute at this pace


def read_env_key(name):
    "The key from the environment, else from a KEY=VALUE line in the gitignored .env at the repository root; None if neither has it. Never printed, never written."
    value = os.environ.get(name)
    if value:
        return value
    env_file = ROOT / ".env"
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            line = line.strip()
            if line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            if k.strip() == name:
                return v.strip().strip('"').strip("'") or None
    return None


LIFI_KEY = read_env_key("LIFI_API_KEY")
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "lifi-takehome-analysis/0.1 (analysis notebook 03)"
if LIFI_KEY:
    SESSION.headers["x-lifi-api-key"] = LIFI_KEY
print("data dir:", DATA_LABEL, "| reference snapshots:", REF.relative_to(ROOT), "in the repository | LI.FI key present:", bool(LIFI_KEY))

RATE = {"calls": 0, "n429": 0, "network_errors": 0, "min_remaining": None, "last_remaining": None}


def now_utc():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def get_json(url, params=None, pause=PACE_SECONDS, max_tries=8, timeout=60):
    "GET with the session headers. Paces at `pause` seconds per call; on HTTP 429 sleeps the ratelimit-reset header (seconds) plus one; on 5xx and network errors backs off exponentially. Returns (status_code, parsed JSON or text, ratelimit_remaining). Status 0 when every try failed on the network."
    wait = 5.0
    r = None
    for attempt in range(max_tries):
        try:
            r = SESSION.get(url, params=params, timeout=timeout)
        except requests.RequestException as e:
            RATE["network_errors"] += 1
            print(f"  network error: {type(e).__name__}; sleeping {wait:.0f}s, attempt {attempt + 1}/{max_tries}")
            time.sleep(wait)
            wait = min(wait * 2, 120)
            if attempt == max_tries - 1:
                return 0, f"network error after {max_tries} tries: {type(e).__name__}", None
            continue
        RATE["calls"] += 1
        remaining = r.headers.get("ratelimit-remaining")
        reset = r.headers.get("ratelimit-reset")
        if remaining is not None and remaining.isdigit():
            RATE["last_remaining"] = int(remaining)
            RATE["min_remaining"] = int(remaining) if RATE["min_remaining"] is None else min(RATE["min_remaining"], int(remaining))
        if r.status_code == 429:
            RATE["n429"] += 1
            sleep_s = float(reset) if reset and reset.replace(".", "", 1).isdigit() else wait
            print(f"  HTTP 429: sleeping {sleep_s:.0f}s (ratelimit-reset), attempt {attempt + 1}/{max_tries}")
            time.sleep(sleep_s + 1)
            continue
        if r.status_code >= 500:
            print(f"  HTTP {r.status_code}: sleeping {wait:.0f}s, attempt {attempt + 1}/{max_tries}")
            time.sleep(wait)
            wait = min(wait * 2, 120)
            continue
        if remaining is not None and remaining.isdigit() and int(remaining) <= 1 and reset and reset.replace(".", "", 1).isdigit():
            time.sleep(float(reset) + 1)   # the window is spent; wait for it to reset rather than take a 429
        elif pause:
            time.sleep(pause)
        try:
            return r.status_code, r.json(), remaining
        except ValueError:
            return r.status_code, r.text, remaining
    return r.status_code, r.text, None


def manifest_read():
    return json.loads(MANIFEST.read_text()) if MANIFEST.exists() else {}


def manifest_write(name, source_url, rows, section, fetched_at, note=None):
    m = manifest_read()
    m[name] = {"source_url": source_url, "fetched_at_utc": fetched_at, "rows": int(rows), "notebook_section": section}
    if note:
        m[name]["note"] = note
    MANIFEST.write_text(json.dumps(m, indent=1, sort_keys=True) + "\n")


def snapshot_csv(name, fetch, section, source_url, note=None, read_kwargs=None, fetched_at=None):
    "Read data/reference/<name> if it exists; otherwise call fetch() -> DataFrame, write it and record the manifest entry. `note` and `fetched_at` may be callables evaluated after the fetch (fetched_at: the time of the first API call behind the file, read from the JSONL, so a re-flatten from the raw responses keeps the original fetch time)."
    path = REF / name
    if path.exists():
        df = pd.read_csv(path, **(read_kwargs or {}))
        print(f"{name}: read snapshot, {len(df)} rows, fetched {manifest_read().get(name, {}).get('fetched_at_utc', '?')} (no network call)")
        return df
    if not LIFI_KEY:
        raise RuntimeError(f"{name} is missing and LIFI_API_KEY is not set: put the key in the environment or in .env, or restore the snapshot")
    print(f"{name}: no snapshot, fetching from {source_url}")
    t0 = now_utc()
    df = fetch()
    df.to_csv(path, index=False)
    t_fetch = fetched_at() if callable(fetched_at) else (fetched_at or t0)
    manifest_write(name, source_url, len(df), section, t_fetch, note() if callable(note) else note)
    print(f"{name}: {len(df)} rows, fetch time recorded {t_fetch} (this cell ran at {t0}), written to {REF.relative_to(ROOT)}/{name} in the repository")
    return pd.read_csv(path, **(read_kwargs or {}))


def wilson(k, n, z=1.96):
    "Wilson 95% interval for k successes of n, as (low, high) rounded to 4 places."
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return round(c - h, 4), round(c + h, 4)


def jsonl_read(path):
    return [json.loads(line) for line in path.read_text().splitlines()] if path.exists() else []


def fetch_log(recs):
    "What the raw JSONL records show about a fetch: calls, final HTTP statuses, first and last call, observed calls per minute, the minimum ratelimit-remaining and the largest gap between consecutive calls (descriptive: pacing plus latency). The JSONL keeps each call's final response only (get_json retries HTTP 429 and 5xx before returning), so a retried 429 is not recorded; the minimum ratelimit-remaining is the evidence on 429s, since one is returned only once the 100-per-minute window is exhausted."
    ts = sorted(pd.Timestamp(r["fetched_at_utc"]) for r in recs)
    gaps = [(b - a).total_seconds() for a, b in zip(ts, ts[1:])]
    rem = [int(r["ratelimit_remaining"]) for r in recs if str(r.get("ratelimit_remaining") or "").isdigit()]
    minutes = max((ts[-1] - ts[0]).total_seconds() / 60, 1e-9) if ts else 1e-9
    return {"calls": len(recs), "http_statuses": dict(sorted(Counter(r["http_status"] for r in recs).items())), "first_call_utc": ts[0].isoformat() if ts else None, "last_call_utc": ts[-1].isoformat() if ts else None,
            "calls_per_minute": round(len(recs) / minutes, 1), "min_ratelimit_remaining": min(rem) if rem else None, "max_gap_between_calls_s": max(gaps) if gaps else None}


def fetch_span(recs):
    "The fetch log of a list of JSONL records as a sentence for the manifest note."
    f = fetch_log(recs)
    return (f"{f['calls']} calls from {f['first_call_utc']} to {f['last_call_utc']} ({f['calls_per_minute']:.0f} per minute observed, paced at {PACE_SECONDS} s sleep after each call plus the call's own latency); "
            f"final HTTP statuses {f['http_statuses']}; minimum ratelimit-remaining {f['min_ratelimit_remaining']} of 100, so the window was never exhausted and no HTTP 429 was returned "
            f"(a retried 429 would not be recorded, the JSONL keeping each call's final response); largest gap between consecutive calls {f['max_gap_between_calls_s']:.0f} s")


def fee_usd_per_unit(line):
    "USD per smallest token unit of a fee line, from its amount and amountUSD; None when either is missing or zero."
    try:
        amt, usd = int(line.get("amount") or 0), float(line.get("amountUSD") or 0)
    except (TypeError, ValueError):
        return None
    return usd / amt if amt > 0 else None


def flatten_status(ev, rec):
    "One flat row for an event row `ev` (dict with the snake_case columns of lifi.load.load_transfers plus amount_usd_provided) and a raw JSONL record `rec` (its `response` is the parsed body)."
    body = rec.get("response")
    out = {"chain": ev["chain"], "tx_hash": ev["tx_hash"], "log_index": int(ev["log_index"]), "block_time": pd.Timestamp(ev["block_time"]).isoformat(),
           "event_transactionId": ev["transaction_id"], "event_bridge": ev["bridge"], "event_integrator": ev["integrator"], "event_sendingAssetId": ev["sending_asset_id"],
           "event_minAmount": str(ev["min_amount"]), "event_destinationChainId": int(ev["destination_chain_id"]), "event_receiver": ev["receiver"], "event_tx_from": ev["tx_from"],
           "event_hasSourceSwaps": bool(ev["has_source_swaps"]), "event_hasDestinationCall": bool(ev["has_destination_call"]), "amount_usd_provided": ev.get("amount_usd_provided"),
           "http_status": rec.get("http_status"), "fetched_at_utc": rec.get("fetched_at_utc")}
    if not isinstance(body, dict):
        out["api_message"] = str(body)[:200]
        return out
    md_ = body.get("metadata") or {}
    out.update({"api_transactionId": body.get("transactionId"), "api_message": body.get("message"), "status": body.get("status"), "substatus": body.get("substatus"),
                "substatusMessage": body.get("substatusMessage"), "tool": body.get("tool"), "fromAddress": body.get("fromAddress"), "toAddress": body.get("toAddress"),
                "metadata_integrator": md_.get("integrator"),
                "transaction_id_matches": (str(body.get("transactionId") or "").lower() == str(ev["transaction_id"]).lower()) if body.get("transactionId") else None})
    for side in ("sending", "receiving"):
        s = body.get(side) or {}
        tok = s.get("token") or {}
        out.update({f"{side}_txHash": s.get("txHash"), f"{side}_chainId": s.get("chainId"), f"{side}_token_address": tok.get("address"),
                    f"{side}_token_symbol": tok.get("symbol"), f"{side}_token_decimals": tok.get("decimals"), f"{side}_amount": s.get("amount"),
                    f"{side}_amountUSD": s.get("amountUSD"), f"{side}_gasAmountUSD": s.get("gasAmountUSD"), f"{side}_timestamp": s.get("timestamp"),
                    f"{side}_included_tools": "|".join(str(st.get("tool")) for st in (s.get("includedSteps") or []))})
    out["sending_hash_matches"] = (str(out.get("sending_txHash") or "").lower() == ev["tx_hash"].lower()) if out.get("sending_txHash") else None
    batch = body.get("batchStatus")   # present on batched (account-abstraction) transactions: one entry per user operation, each with its own `calls`, every call a full transfer record
    if isinstance(batch, list):
        calls = [c for e in batch for c in (e.get("calls") or []) if isinstance(c, dict)]
        out.update({"batch_n": len(batch), "batch_success": "|".join(str(e.get("success")) for e in batch), "batch_calls_n": len(calls),
                    "batch_call_transactionIds": "|".join(str(c.get("transactionId")) for c in calls),
                    "batch_call_statuses": "|".join(f"{c.get('status')}/{c.get('substatus')}" for c in calls),
                    "batch_call_tools": "|".join(str(c.get("tool")) for c in calls)})
    else:
        out.update({"batch_n": None, "batch_success": None, "batch_calls_n": None, "batch_call_transactionIds": None, "batch_call_statuses": None, "batch_call_tools": None})
    q = body.get("quote") or {}
    est = q.get("estimate") or {}
    fees, gas = est.get("feeCosts") or [], est.get("gasCosts") or []
    usd = lambda items: sum(float(i.get("amountUSD") or 0) for i in items)
    lifi_lines = [f for f in fees if str(f.get("name", "")).startswith("LIFI")]
    other_lines = [f for f in fees if not str(f.get("name", "")).startswith("LIFI")]
    split_lifi = split_integ = 0.0
    for f in lifi_lines:
        per_unit, fs = fee_usd_per_unit(f), f.get("feeSplit") or {}
        if per_unit is not None:
            split_lifi += int(fs.get("lifiFee") or 0) * per_unit
            split_integ += int(fs.get("integratorFee") or 0) * per_unit
    out.update({"quote_tool": q.get("tool"), "quote_integrator": q.get("integrator"), "quote_fromAmountUSD": est.get("fromAmountUSD"), "quote_toAmountUSD": est.get("toAmountUSD"),
                "quote_fee_usd": usd(fees) if fees else None,
                "quote_fee_lifi_usd": usd(lifi_lines) if fees else None,             # fee lines whose name starts with "LIFI"
                "quote_fee_other_usd": usd(other_lines) if fees else None,
                "quote_fee_names": "|".join(str(f.get("name")) for f in fees),
                "quote_fee_lifi_percentages": "|".join(str(f.get("percentage")) for f in lifi_lines),
                "quote_fee_lifi_included": "|".join(str(f.get("included")) for f in lifi_lines),
                "fee_split_lifi_usd": split_lifi if lifi_lines else None,            # feeSplit.lifiFee of the LIFI-named lines, in USD at the line's own rate
                "fee_split_integrator_usd": split_integ if lifi_lines else None,     # feeSplit.integratorFee likewise
                "fee_split_recipients": "|".join(str(r.get("name")) for f in lifi_lines for r in ((f.get("feeSplit") or {}).get("recipients") or [])),
                "quote_gas_usd": usd(gas) if gas else None})
    return out


STR_COLS = ["tx_hash", "event_transactionId", "event_sendingAssetId", "event_minAmount", "event_receiver", "event_tx_from", "api_transactionId", "sending_txHash", "receiving_txHash",
            "sending_token_address", "receiving_token_address", "sending_amount", "receiving_amount", "fromAddress", "toAddress", "status", "substatus", "substatusMessage",
            "tool", "quote_tool", "quote_fee_names", "quote_fee_lifi_percentages", "quote_fee_lifi_included", "fee_split_recipients", "sending_included_tools", "receiving_included_tools",
            "batch_success", "batch_call_transactionIds", "batch_call_statuses", "batch_call_tools", "reason_tags", "lookup", "lookup_value", "bridge_param", "sample_design", "api_message", "metadata_integrator", "quote_integrator"]
NUM_COLS = ["sending_amountUSD", "receiving_amountUSD", "sending_gasAmountUSD", "receiving_gasAmountUSD", "quote_fromAmountUSD", "quote_toAmountUSD", "quote_fee_usd",
            "quote_fee_lifi_usd", "quote_fee_other_usd", "fee_split_lifi_usd", "fee_split_integrator_usd", "quote_gas_usd", "amount_usd_provided"]


def read_status_csv(name):
    df = pd.read_csv(REF / name, dtype={c: str for c in STR_COLS if True}, keep_default_na=True)
    for c in NUM_COLS:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ("sending_chainId", "receiving_chainId", "batch_n", "batch_calls_n", "log_index", "event_destinationChainId", "http_status"):
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    return df


data dir: LIFI_DATA_DIR | reference snapshots: data/reference in the repository | LI.FI key present: False


## 2. The data, the provided-feed USD column and the 600-event snapshot

In [2]:
tf = load_transfers()
prices = load_prices()
tf["hour"] = tf["block_time"].dt.floor("h")
joined = tf.merge(prices[["chain", "token_address", "hour", "decimals", "price_usd"]], left_on=["chain", "sending_asset_id", "hour"], right_on=["chain", "token_address", "hour"], how="left")


def token_amount(min_amount, decimals):
    return float("nan") if pd.isna(decimals) else float(Decimal(int(min_amount)) / Decimal(10) ** int(decimals))


tf["amount_token"] = pd.Series([token_amount(a, d) for a, d in zip(joined["min_amount"], joined["decimals"])], index=joined.index)
tf["amount_usd_provided"] = tf["amount_token"] * joined["price_usd"]
tf["key"] = list(zip(tf["chain"], tf["tx_hash"], tf["log_index"]))
assert len(tf) == 124_858 and tf["key"].is_unique
print(f"transfers {tf.shape}; priced by the provided feed {int(tf['amount_usd_provided'].notna().sum())}, unpriced {int(tf['amount_usd_provided'].isna().sum())}, total USD {tf['amount_usd_provided'].sum():,.2f}")

s600 = read_status_csv("lifi_api_transfer_status_sample_600.csv")
s600_raw = {(r["chain"], r["tx_hash"], int(r["log_index"])): r for r in jsonl_read(REF / "lifi_api_transfer_status_sample_600_raw.jsonl")}
K600 = set(zip(s600["chain"], s600["tx_hash"], s600["log_index"].astype(int)))
assert len(s600) == 600 and len(s600_raw) == 600 and K600 == set(s600_raw)
print("600-event snapshot (01 §5):", s600["chain"].value_counts().to_dict(), " status/substatus:", s600.groupby(["status", "substatus"]).size().to_dict())


transfers (124858, 21); priced by the provided feed 124477, unpriced 381, total USD 307,935,992.59
600-event snapshot (01 §5): {'arbitrum': 100, 'base': 100, 'bnb': 100, 'ethereum': 100, 'optimism': 100, 'polygon': 100}  status/substatus: {('DONE', 'COMPLETED'): 593, ('DONE', 'PARTIAL'): 1, ('DONE', 'REFUNDED'): 6}


## 3. Edge-case selection

Rows the profile singled out. A row can carry several reason tags. Random picks use `DataFrame.sample(random_state=20261029)` over the rows of each pool in file order; the pools are not reduced by the 600 already sampled (an overlap, if any, is refetched and flagged `in_sample_600`).

In [3]:
UNKNOWN_BIG = 1885080386571452                                  # 685 rows, not in LI.FI's chain list, every row with the receiver sentinel (01 §7)
UNKNOWN_OTHER = [81457, 204, 60808, 1201081091099710, 232]       # the other five ids not in the chain list, 34 rows together
NON_EVM = {"solana": [1151111081099710], "utxo": [20000000000001, 20000000000005], "sui": [9270000000000000], "tron": [728126428]}   # 01 §7: SVM, UTXO (Bitcoin, Zcash), MVM, TVM

reasons = {}
id_counts = tf["transaction_id"].value_counts()
DUP_IDS = sorted(id_counts[id_counts > 1].index)
reasons["dup_transaction_id"] = tf.index[tf["transaction_id"].isin(DUP_IDS)]
events_per_hash = tf.groupby(["chain", "tx_hash"])["log_index"].transform("size")
reasons["two_event_tx"] = tf.index[events_per_hash > 1]
reasons["above_1m_usd"] = tf.index[tf["amount_usd_provided"] > 1_000_000]
reasons["round_trip_9519447"] = tf.index[(tf["min_amount"] == 9_519_447_993_443) & (tf["integrator"] == "_arbitrum")]   # 9,519,447.993443 USDC, once in each direction (profile §11)
reasons["empty_integrator"] = tf.index[tf["integrator"] == ""]
reasons["unknown_dest_1885080386571452"] = tf[tf["destination_chain_id"] == UNKNOWN_BIG].sample(n=50, random_state=SEED).index
reasons["unknown_dest_other"] = tf.index[tf["destination_chain_id"].isin(UNKNOWN_OTHER)]
sentinel = tf[tf["receiver"] == SENTINEL_RECEIVER]
for kind, ids in NON_EVM.items():
    pool = sentinel[sentinel["destination_chain_id"].isin(ids)]
    reasons[f"sentinel_{kind}"] = pool.sample(n=min(50, len(pool)), random_state=SEED).index
    print(f"sentinel_{kind}: pool {len(pool)} rows to {ids}, picked {min(50, len(pool))}")

tags = defaultdict(list)
for tag, idx in reasons.items():
    for i in idx:
        tags[i].append(tag)
edge = tf.loc[sorted(tags)].copy()
edge["reason_tags"] = ["|".join(tags[i]) for i in edge.index]
edge["in_sample_600"] = edge["key"].isin(K600)
edge["n_reasons"] = edge["reason_tags"].str.count(r"\|") + 1
summary = pd.DataFrame([{"reason": tag, "rows": len(idx), "distinct_tx_hashes": tf.loc[idx, ["chain", "tx_hash"]].drop_duplicates().shape[0],
                         "rows_with_another_reason": int((edge.loc[idx, "n_reasons"] > 1).sum()), "rows_in_sample_600": int(edge.loc[idx, "in_sample_600"].sum()),
                         "usd_provided": tf.loc[idx, "amount_usd_provided"].sum()} for tag, idx in reasons.items()]).set_index("reason")
print(f"edge-case rows: {len(edge)} (sum of reason rows {summary['rows'].sum()}, rows with more than one reason {int((edge['n_reasons'] > 1).sum())}); distinct (chain, tx_hash): {edge[['chain', 'tx_hash']].drop_duplicates().shape[0]}; in the 600 sample: {int(edge['in_sample_600'].sum())}")
display(summary.round(2))
print("rows carrying more than one reason:")
display(edge[edge["n_reasons"] > 1][["chain", "block_time", "tx_hash", "log_index", "bridge", "integrator", "destination_chain_id", "amount_usd_provided", "reason_tags"]].round({"amount_usd_provided": 2}))
print("edge rows by chain:", edge["chain"].value_counts().to_dict())


sentinel_solana: pool 14229 rows to [1151111081099710], picked 50
sentinel_utxo: pool 1581 rows to [20000000000001, 20000000000005], picked 50
sentinel_sui: pool 167 rows to [9270000000000000], picked 50
sentinel_tron: pool 2 rows to [728126428], picked 2
edge-case rows: 294 (sum of reason rows 296, rows with more than one reason 2); distinct (chain, tx_hash): 283; in the 600 sample: 1


,rows,distinct_tx_hashes,rows_with_another_reason,rows_in_sample_600,usd_provided
reason,,,,,
dup_transaction_id,12,12,0,0,50357.82
two_event_tx,22,11,0,0,7744.05
above_1m_usd,22,22,2,1,76115424.09
round_trip_9519447,2,2,2,0,19042370.59
empty_integrator,2,2,0,0,3.09
unknown_dest_1885080386571452,50,50,0,0,529201.37
unknown_dest_other,34,34,0,0,2509.11
sentinel_solana,50,50,0,0,29968.89
sentinel_utxo,50,50,0,0,55661.01


rows carrying more than one reason:


,chain,block_time,tx_hash,log_index,bridge,integrator,destination_chain_id,amount_usd_provided,reason_tags
102526,ethereum,2026-10-18 17:01:59,0x08f630e8c0afd0d8f9a9af9af6c82d2e908713534fecc858ba8fe947fbf8bb26,336,polymerStandard,_arbitrum,42161,9521501.02,above_1m_usd|round_trip_9519447
120301,arbitrum,2026-10-19 17:57:04,0x436b3698f1db02d8e5c9391310f19e9abf87e336f96bd185b3e5ed52dfb162a9,16,polymerStandard,_arbitrum,1,9520869.56,above_1m_usd|round_trip_9519447


edge rows by chain: {'ethereum': 130, 'base': 60, 'arbitrum': 50, 'bnb': 38, 'polygon': 14, 'optimism': 2}


## 4. Fetch: the edge cases

One `/v1/status?txHash=<tx_hash>` call per distinct (chain, tx_hash) of the edge rows (lookup `tx_hash`); for the 22 rows of the two-event transactions one further call per row with `&bridge=<event bridge>` (lookup `tx_hash_bridge`), to see whether the parameter selects the leg; and for the 6 duplicate `transactionId` values one call with the id itself as `txHash` (lookup `transaction_id`), which the endpoint accepts, to see which of the two transactions LI.FI maps the id to. Raw responses go to `lifi_api_transfer_status_edge_cases_raw.jsonl` as they arrive (one line per call, keyed by lookup, value and bridge parameter), so a timeout resumes; the flattened CSV has one row per (edge row, lookup).

In [4]:
EDGE_CSV, EDGE_JSONL = "lifi_api_transfer_status_edge_cases.csv", REF / "lifi_api_transfer_status_edge_cases_raw.jsonl"
EDGE_URL = f"{LIFI}/status?txHash=<tx_hash> (lookup tx_hash), {LIFI}/status?txHash=<tx_hash>&bridge=<bridge> (lookup tx_hash_bridge), {LIFI}/status?txHash=<transactionId> (lookup transaction_id)"


def edge_calls():
    "The planned calls: (lookup, chain, lookup_value, bridge_param, params)."
    calls = []
    for chain, h in edge[["chain", "tx_hash"]].drop_duplicates().itertuples(index=False):
        calls.append({"lookup": "tx_hash", "chain": chain, "lookup_value": h, "bridge_param": None, "params": {"txHash": h}})
    for ev in edge[edge["reason_tags"].str.contains("two_event_tx")].itertuples(index=False):
        calls.append({"lookup": "tx_hash_bridge", "chain": ev.chain, "lookup_value": ev.tx_hash, "bridge_param": ev.bridge, "params": {"txHash": ev.tx_hash, "bridge": ev.bridge}})
    for tid in DUP_IDS:
        chain = tf.loc[tf["transaction_id"] == tid, "chain"].iloc[0]
        calls.append({"lookup": "transaction_id", "chain": chain, "lookup_value": tid, "bridge_param": None, "params": {"txHash": tid}})
    return calls


def call_key(c):
    return (c["lookup"], c["lookup_value"], c.get("bridge_param"))


def fetch_calls(calls, jsonl_path, extra=None, label=""):
    "Run every call not yet in jsonl_path, appending one record per response as it arrives. Returns {key: record}."
    done = {call_key(r): r for r in jsonl_read(jsonl_path)}
    todo = [c for c in calls if call_key(c) not in done]
    print(f"{label}: {len(calls)} calls planned, {len(done)} already in {jsonl_path.name}, {len(todo)} to fetch (~{len(todo) * PACE_SECONDS / 60:.1f} min)")
    t_start = time.time()
    with jsonl_path.open("a") as fh:
        for i, c in enumerate(todo):
            status, body, remaining = get_json(f"{LIFI}/status", params=c["params"])
            rec = {"lookup": c["lookup"], "chain": c["chain"], "lookup_value": c["lookup_value"], "bridge_param": c.get("bridge_param"), "request_params": c["params"],
                   "fetched_at_utc": now_utc(), "http_status": status, "ratelimit_remaining": remaining, "response": body}
            if extra:
                rec.update(extra(c))
            fh.write(json.dumps(rec) + "\n")
            fh.flush()
            done[call_key(c)] = rec
            if (i + 1) % 200 == 0 or i + 1 == len(todo):
                print(f"  {i + 1}/{len(todo)} fetched, {(time.time() - t_start) / 60:.1f} min, HTTP 429 so far {RATE['n429']}, min ratelimit-remaining seen {RATE['min_remaining']}")
    return done


def flatten_edge(ev, rec, lookup, bridge_param=None):
    row = flatten_status(ev, rec)
    row.update({"reason_tags": ev["reason_tags"], "in_sample_600": bool(ev["in_sample_600"]), "lookup": lookup, "lookup_value": rec["lookup_value"], "bridge_param": bridge_param})
    return row


def fetch_edge_cases():
    done = fetch_calls(edge_calls(), EDGE_JSONL, label="edge cases")
    rows = []
    for ev in edge.to_dict("records"):
        rows.append(flatten_edge(ev, done[("tx_hash", ev["tx_hash"], None)], "tx_hash"))
        if "two_event_tx" in ev["reason_tags"]:
            rows.append(flatten_edge(ev, done[("tx_hash_bridge", ev["tx_hash"], ev["bridge"])], "tx_hash_bridge", ev["bridge"]))
        if "dup_transaction_id" in ev["reason_tags"]:
            rows.append(flatten_edge(ev, done[("transaction_id", ev["transaction_id"], None)], "transaction_id"))
    out = pd.DataFrame(rows)
    out["lookup_order"] = out["lookup"].map({"tx_hash": 0, "tx_hash_bridge": 1, "transaction_id": 2})
    return out.sort_values(["lookup_order", "chain", "block_time", "log_index"]).drop(columns="lookup_order").reset_index(drop=True)


def edge_note():
    recs = jsonl_read(EDGE_JSONL)
    by_lookup = Counter(r["lookup"] for r in recs)
    return (f"{len(recs)} calls: {dict(by_lookup)}; {fetch_span(recs)}; {len(edge)} edge rows over {len(reasons)} reasons, "
            f"random picks with DataFrame.sample(random_state={SEED}); one CSV row per (edge row, lookup); raw responses in lifi_api_transfer_status_edge_cases_raw.jsonl")


edge_status = snapshot_csv(EDGE_CSV, fetch_edge_cases, "03 §4", EDGE_URL, note=edge_note, fetched_at=lambda: min(r["fetched_at_utc"] for r in jsonl_read(EDGE_JSONL)))
edge_status = read_status_csv(EDGE_CSV)
manifest = manifest_read()
if EDGE_JSONL.name not in manifest:
    manifest_write(EDGE_JSONL.name, EDGE_URL, sum(1 for _ in EDGE_JSONL.open()), "03 §4", manifest[EDGE_CSV]["fetched_at_utc"],
                   note="raw JSON responses, one line per call, with lookup kind, value, bridge parameter, request parameters (no key), HTTP status and ratelimit-remaining")
print(f"{EDGE_CSV}: {edge_status.shape}; rows by lookup {edge_status['lookup'].value_counts().to_dict()}; HTTP statuses {edge_status['http_status'].value_counts().to_dict()}")
print("status/substatus of the tx_hash lookups:", edge_status[edge_status["lookup"] == "tx_hash"].groupby(["status", "substatus"], dropna=False).size().to_dict())


lifi_api_transfer_status_edge_cases.csv: read snapshot, 328 rows, fetched 2026-10-28T19:01:00+00:00 (no network call)
lifi_api_transfer_status_edge_cases.csv: (328, 73); rows by lookup {'tx_hash': 294, 'tx_hash_bridge': 22, 'transaction_id': 12}; HTTP statuses {np.int64(200): 328}
status/substatus of the tx_hash lookups: {('DONE', 'COMPLETED'): 285, ('DONE', 'PARTIAL'): 3, ('DONE', 'REFUNDED'): 6}


## 5. Edge-case results

One subsection per reason. `ex` is the plain hash lookup (`lookup = tx_hash`) of every edge row; the bridge-parameter and id lookups are read where the subsection needs them. `completed` means `status = DONE` and `substatus = COMPLETED`.

In [5]:
ex = edge_status[edge_status["lookup"] == "tx_hash"].copy()
ex["completed"] = (ex["status"] == "DONE") & (ex["substatus"] == "COMPLETED")
ex["dest_chain_matches"] = ex["receiving_chainId"] == ex["event_destinationChainId"]
assert len(ex) == len(edge) and set(zip(ex["chain"], ex["tx_hash"], ex["log_index"].astype(int))) == set(edge["key"])
print("edge rows by HTTP status:", ex["http_status"].value_counts().to_dict())
print("status/substatus:", ex.groupby(["status", "substatus"], dropna=False).size().to_dict())
print("identity: sending hash matches", int(ex["sending_hash_matches"].fillna(False).astype(bool).sum()), "| transactionId matches", int(ex["transaction_id_matches"].fillna(False).astype(bool).sum()),
      "| integrator matches", int((ex["metadata_integrator"].fillna("") == ex["event_integrator"].fillna("")).sum()), "(empty strings compared as empty)", "| tool equals bridge string", int((ex["tool"] == ex["event_bridge"]).sum()), "of", len(ex))
per_reason = pd.DataFrame([{"reason": tag, "rows": int(ex["reason_tags"].str.contains(tag).sum()), "completed": int(ex[ex["reason_tags"].str.contains(tag)]["completed"].sum()),
                            "refunded": int((ex[ex["reason_tags"].str.contains(tag)]["substatus"] == "REFUNDED").sum()), "partial": int((ex[ex["reason_tags"].str.contains(tag)]["substatus"] == "PARTIAL").sum()),
                            "not_done": int((ex[ex["reason_tags"].str.contains(tag)]["status"] != "DONE").sum()), "transactionId_matches": int(ex[ex["reason_tags"].str.contains(tag)]["transaction_id_matches"].fillna(False).astype(bool).sum()),
                            "dest_chain_matches": int(ex[ex["reason_tags"].str.contains(tag)]["dest_chain_matches"].fillna(False).astype(bool).sum())} for tag in reasons]).set_index("reason")
display(per_reason)
edge_raw = {(r["lookup"], r["lookup_value"], r.get("bridge_param")): r for r in jsonl_read(EDGE_JSONL)}
print("transactionId mismatches outside the two-event transactions (the response describes another transfer than the event):")
display(ex[~ex["transaction_id_matches"].fillna(False).astype(bool) & ~ex["reason_tags"].str.contains("two_event_tx")][["chain", "tx_hash", "log_index", "reason_tags", "event_bridge", "event_integrator", "event_transactionId", "api_transactionId", "tool", "status", "substatus", "batch_n", "batch_calls_n", "batch_call_transactionIds"]])
print("responses carrying batchStatus:", int(ex["batch_n"].notna().sum()), "of", len(ex), "; by (entries, calls):", ex[ex["batch_n"].notna()].groupby(["batch_n", "batch_calls_n"]).size().to_dict())

edge rows by HTTP status: {np.int64(200): 294}
status/substatus: {('DONE', 'COMPLETED'): 285, ('DONE', 'PARTIAL'): 3, ('DONE', 'REFUNDED'): 6}
identity: sending hash matches 294 | transactionId matches 282 | integrator matches 294 (empty strings compared as empty) | tool equals bridge string 288 of 294


,rows,completed,refunded,partial,not_done,transactionId_matches,dest_chain_matches
reason,,,,,,,
dup_transaction_id,12,8,4,0,0,12,8
two_event_tx,22,22,0,0,0,11,16
above_1m_usd,22,22,0,0,0,22,22
round_trip_9519447,2,2,0,0,0,2,2
empty_integrator,2,2,0,0,0,2,2
unknown_dest_1885080386571452,50,50,0,0,0,50,0
unknown_dest_other,34,34,0,0,0,34,34
sentinel_solana,50,48,2,0,0,49,48
sentinel_utxo,50,50,0,0,0,50,50


transactionId mismatches outside the two-event transactions (the response describes another transfer than the event):


,chain,tx_hash,log_index,reason_tags,event_bridge,event_integrator,event_transactionId,api_transactionId,tool,status,substatus,batch_n,batch_calls_n,batch_call_transactionIds
78,base,0xad02b758c2342184d0e167e3d968422f773bb76d63370e16c899e0dd91097b75,877,sentinel_solana,mayan,base-app,0x9d62702c04bdf22f1ef8a82c4dc14df734dbd80f362c7b8f90f81cdb28cae764,0x5b1c36c321c012401c260fd1b5f38747507b637da8ddba079b395759c9b9ca28,mayan,DONE,COMPLETED,2,2,0x9d62702c04bdf22f1ef8a82c4dc14df734dbd80f362c7b8f90f81cdb28cae764...


responses carrying batchStatus: 45 of 294 ; by (entries, calls): {(1, 1): 24, (1, 2): 2, (2, 2): 19}


### 5.1 Duplicate `transactionId` pairs: one transfer or two?

In [6]:
# 5.1 Duplicate transactionId pairs: the hash lookup of each of the two transactions, and the id lookup.
cols = ["chain", "block_time", "tx_hash", "log_index", "event_bridge", "event_integrator", "sending_token_symbol", "sending_amountUSD", "api_transactionId", "transaction_id_matches", "sending_hash_matches",
        "status", "substatus", "tool", "receiving_chainId", "receiving_token_symbol", "receiving_amountUSD", "receiving_txHash", "batch_n"]
dups = ex[ex["reason_tags"].str.contains("dup_transaction_id")].sort_values(["event_transactionId", "block_time"])
dups["pair"] = dups.groupby("event_transactionId").ngroup() + 1
dups["leg"] = dups.groupby("event_transactionId").cumcount() + 1
print("hash lookups of the 12 rows (pair = the shared transactionId, leg = order by block_time):")
display(dups[["pair", "leg"] + cols].round(2))
print("per pair: are the two responses the same record (same sending txHash, same receiving txHash) or two?")
pairs = dups.groupby("pair").agg(transactionId=("event_transactionId", "first"), chain=("chain", "first"), bridge=("event_bridge", "first"), integrator=("event_integrator", "first"),
                                 seconds_apart=("block_time", lambda s: (pd.to_datetime(s).max() - pd.to_datetime(s).min()).total_seconds()),
                                 distinct_sending_txHash=("sending_txHash", "nunique"), distinct_receiving_txHash=("receiving_txHash", "nunique"), distinct_api_transactionId=("api_transactionId", "nunique"),
                                 both_id_match=("transaction_id_matches", lambda s: bool(s.fillna(False).astype(bool).all())), both_hash_match=("sending_hash_matches", lambda s: bool(s.fillna(False).astype(bool).all())),
                                 statuses=("substatus", lambda s: "|".join(map(str, s))), sending_usd=("sending_amountUSD", lambda s: "|".join(f"{v:.2f}" for v in s)), receiving_usd=("receiving_amountUSD", lambda s: "|".join(f"{v:.2f}" for v in s)))
display(pairs)
idl = edge_status[edge_status["lookup"] == "transaction_id"].sort_values(["event_transactionId", "block_time"]).copy()
idl["leg"] = idl.groupby("event_transactionId").cumcount() + 1
print("the id lookup (txHash=<transactionId>), one response per id, shown against each of the two rows: which transaction's hash does the response carry?")
display(idl[["leg", "chain", "block_time", "tx_hash", "log_index", "http_status", "api_transactionId", "sending_txHash", "sending_hash_matches", "status", "substatus", "sending_amountUSD", "receiving_amountUSD"]].round(2))
print("id lookup resolves to leg:", idl[idl["sending_hash_matches"].fillna(False).astype(bool)].groupby("event_transactionId")["leg"].first().value_counts().to_dict(), "(1 = earlier transaction, 2 = later)")

hash lookups of the 12 rows (pair = the shared transactionId, leg = order by block_time):


,pair,leg,chain,block_time,tx_hash,log_index,event_bridge,event_integrator,sending_token_symbol,sending_amountUSD,api_transactionId,transaction_id_matches,sending_hash_matches,status,substatus,tool,receiving_chainId,receiving_token_symbol,receiving_amountUSD,receiving_txHash,batch_n
278,1,1,optimism,2026-10-13T05:52:31,0x8388889759abc3ef731d501ee560ae82c31cdd9ce963c91e568a9e684b0bb002,287,relaydepository,epoch,USDT,0.10,0x46b6ede720d2c6ac3d1e9a163942f3c03bf72ceb20794574c4dc285415e9668a,True,True,DONE,COMPLETED,relaydepository,1,DAI,0.06,0x620ee9f35cfada355a8b001d25c6bd500ec65890814df5c42be762463819c732,<NA>
279,1,2,optimism,2026-10-13T06:00:29,0x07dfea310ddc4755e523ad42958381acd1738b70eb622652c37e299e1172df85,120,relaydepository,epoch,USDT,0.10,0x46b6ede720d2c6ac3d1e9a163942f3c03bf72ceb20794574c4dc285415e9668a,True,True,DONE,REFUNDED,relaydepository,10,USDT,0.10,0x7af93469cba0fed57cda4f22169700f0852035e71aef4fc3ab1eb1efc62e416a,<NA>
118,2,1,bnb,2026-10-13T11:07:02,0x60f253000cc123b6ddf4985db78953994ceec39b6266a3fff61162fae780b4cb,557,relaydepository,sushi,USDT,49.98,0x4b888aebfd3b57fc129333c1c9acc74a61cee1693fab85f17fc21b2cef6e7c2e,True,True,DONE,COMPLETED,relaydepository,324,XVS,48.31,0x1bae0cfec5adf4fc817135788b655dde909d5a504a1232475001d016883c644f,<NA>
119,2,2,bnb,2026-10-13T11:08:04,0xbbfe9f831e46648d049046162a547c4a229e718c159165d16fc611bbaa4883d1,311,relaydepository,sushi,USDT,49.98,0x4b888aebfd3b57fc129333c1c9acc74a61cee1693fab85f17fc21b2cef6e7c2e,True,True,DONE,REFUNDED,relaydepository,56,USDe,49.78,0x581d5ace03bcaa43931c11e53ae54d8b5939921745d6068dc8a72f34c908f8c9,<NA>
5,3,1,arbitrum,2026-10-14T08:49:56,0x340cd5eacb476dad862c5156eb8dc78d6721e0bc5cd0a447fa10e8db17c17f86,5,across,jumper.exchange,ETH,25057.60,0x7127c2c539d3db9aa9e4d4eead797492b907b79dc8d7969809f456cb46b3a24c,True,True,DONE,COMPLETED,across,1,ETH,25026.69,0xb80b3214ad860484d4ba34ab75c931f3d448c7409a9e986056b4b302c163cd15,<NA>
6,3,2,arbitrum,2026-10-14T08:50:14,0xddb372bd807764085dba892488487bb563e2567254a0dcf745f7ba25c7ed042b,5,across,jumper.exchange,ETH,25042.50,0x7127c2c539d3db9aa9e4d4eead797492b907b79dc8d7969809f456cb46b3a24c,True,True,DONE,COMPLETED,across,1,ETH,25031.59,0x69359aadc87c43bc3d6c3ed783dc7884b3a08837c915f076713cc6f2e6e6e4e0,<NA>
41,4,1,arbitrum,2026-10-19T14:50:21,0x5049d8d9f770ce76897ac019d1971ab34e3db215c0ce647cafedfabb81390beb,3,relaydepository,Barker,ETH,2.41,0x858323ababaab6995d2b8d768d95ae9ce0bafeefbc83bbcbf5581513a35c81a9,True,True,DONE,COMPLETED,relaydepository,999,USDC,2.36,0xac932eb4ee41bd8ac0a67cf5b0b6a6c92cf15d5f49372864f402cbe646a76d0e,<NA>
42,4,2,arbitrum,2026-10-19T14:50:25,0x57a44a78a9c0ec9348d4f53c0b5eddbaff4876b22af291523fd7bb162a8ee90e,3,relaydepository,Barker,ETH,2.41,0x858323ababaab6995d2b8d768d95ae9ce0bafeefbc83bbcbf5581513a35c81a9,True,True,DONE,REFUNDED,relaydepository,42161,ETH,2.40,0x42938561543dee84231e86ace98afe6603e7222c866a3276344e09e593b2ef9c,<NA>
275,5,1,ethereum,2026-10-19T20:41:23,0xbe72462d374d4d1f89c64fb4f97ee6fa4b067711de7c775ff9a21a89176ec865,336,gasZipBridge,qern,ETH,0.02,0x96d54526c03173ee1c8f37e28de93deeb59a37eabc0d8c6288c370be7f639db4,True,True,DONE,COMPLETED,gasZipBridge,8453,ETH,0.02,0xef5ad5c3a658950091d6dd44cf1e32182acb2eaa33e95b36246a491a9743d591,<NA>
276,5,2,ethereum,2026-10-19T20:46:11,0x8cac270c2ecea6b97850417748cb828ad9c4d831f4d0827f47e244cf0acef88b,299,gasZipBridge,qern,ETH,0.02,0x96d54526c03173ee1c8f37e28de93deeb59a37eabc0d8c6288c370be7f639db4,True,True,DONE,COMPLETED,gasZipBridge,8453,ETH,0.02,0x64cf2127b6b198f02c6ab926c044649ee98fc968cfa2204cf2d112a1b9d26f6c,<NA>


per pair: are the two responses the same record (same sending txHash, same receiving txHash) or two?


,transactionId,chain,bridge,integrator,seconds_apart,distinct_sending_txHash,distinct_receiving_txHash,distinct_api_transactionId,both_id_match,both_hash_match,statuses,sending_usd,receiving_usd
pair,,,,,,,,,,,,,
1,0x46b6ede720d2c6ac3d1e9a163942f3c03bf72ceb20794574c4dc285415e9668a,optimism,relaydepository,epoch,478.0,2,2,1,True,True,COMPLETED|REFUNDED,0.10|0.10,0.06|0.10
2,0x4b888aebfd3b57fc129333c1c9acc74a61cee1693fab85f17fc21b2cef6e7c2e,bnb,relaydepository,sushi,62.0,2,2,1,True,True,COMPLETED|REFUNDED,49.98|49.98,48.31|49.78
3,0x7127c2c539d3db9aa9e4d4eead797492b907b79dc8d7969809f456cb46b3a24c,arbitrum,across,jumper.exchange,18.0,2,2,1,True,True,COMPLETED|COMPLETED,25057.60|25042.50,25026.69|25031.59
4,0x858323ababaab6995d2b8d768d95ae9ce0bafeefbc83bbcbf5581513a35c81a9,arbitrum,relaydepository,Barker,4.0,2,2,1,True,True,COMPLETED|REFUNDED,2.41|2.41,2.36|2.40
5,0x96d54526c03173ee1c8f37e28de93deeb59a37eabc0d8c6288c370be7f639db4,ethereum,gasZipBridge,qern,288.0,2,2,1,True,True,COMPLETED|COMPLETED,0.02|0.02,0.02|0.02
6,0xee199f180cc6ca794e94e368ccd676ad714bb39406b07ec97966948787af9b02,bnb,relaydepository,Barker,7.0,2,2,1,True,True,COMPLETED|REFUNDED,2.18|2.18,2.14|2.17


the id lookup (txHash=<transactionId>), one response per id, shown against each of the two rows: which transaction's hash does the response carry?


,leg,chain,block_time,tx_hash,log_index,http_status,api_transactionId,sending_txHash,sending_hash_matches,status,substatus,sending_amountUSD,receiving_amountUSD
326,1,optimism,2026-10-13T05:52:31,0x8388889759abc3ef731d501ee560ae82c31cdd9ce963c91e568a9e684b0bb002,287,200,0x46b6ede720d2c6ac3d1e9a163942f3c03bf72ceb20794574c4dc285415e9668a,0x8388889759abc3ef731d501ee560ae82c31cdd9ce963c91e568a9e684b0bb002,True,DONE,COMPLETED,0.10,0.06
327,2,optimism,2026-10-13T06:00:29,0x07dfea310ddc4755e523ad42958381acd1738b70eb622652c37e299e1172df85,120,200,0x46b6ede720d2c6ac3d1e9a163942f3c03bf72ceb20794574c4dc285415e9668a,0x8388889759abc3ef731d501ee560ae82c31cdd9ce963c91e568a9e684b0bb002,False,DONE,COMPLETED,0.10,0.06
320,1,bnb,2026-10-13T11:07:02,0x60f253000cc123b6ddf4985db78953994ceec39b6266a3fff61162fae780b4cb,557,200,0x4b888aebfd3b57fc129333c1c9acc74a61cee1693fab85f17fc21b2cef6e7c2e,0x60f253000cc123b6ddf4985db78953994ceec39b6266a3fff61162fae780b4cb,True,DONE,COMPLETED,49.98,48.31
321,2,bnb,2026-10-13T11:08:04,0xbbfe9f831e46648d049046162a547c4a229e718c159165d16fc611bbaa4883d1,311,200,0x4b888aebfd3b57fc129333c1c9acc74a61cee1693fab85f17fc21b2cef6e7c2e,0x60f253000cc123b6ddf4985db78953994ceec39b6266a3fff61162fae780b4cb,False,DONE,COMPLETED,49.98,48.31
316,1,arbitrum,2026-10-14T08:49:56,0x340cd5eacb476dad862c5156eb8dc78d6721e0bc5cd0a447fa10e8db17c17f86,5,200,0x7127c2c539d3db9aa9e4d4eead797492b907b79dc8d7969809f456cb46b3a24c,0x340cd5eacb476dad862c5156eb8dc78d6721e0bc5cd0a447fa10e8db17c17f86,True,DONE,COMPLETED,25057.60,25026.69
317,2,arbitrum,2026-10-14T08:50:14,0xddb372bd807764085dba892488487bb563e2567254a0dcf745f7ba25c7ed042b,5,200,0x7127c2c539d3db9aa9e4d4eead797492b907b79dc8d7969809f456cb46b3a24c,0x340cd5eacb476dad862c5156eb8dc78d6721e0bc5cd0a447fa10e8db17c17f86,False,DONE,COMPLETED,25057.60,25026.69
318,1,arbitrum,2026-10-19T14:50:21,0x5049d8d9f770ce76897ac019d1971ab34e3db215c0ce647cafedfabb81390beb,3,200,0x858323ababaab6995d2b8d768d95ae9ce0bafeefbc83bbcbf5581513a35c81a9,0x5049d8d9f770ce76897ac019d1971ab34e3db215c0ce647cafedfabb81390beb,True,DONE,COMPLETED,2.41,2.36
319,2,arbitrum,2026-10-19T14:50:25,0x57a44a78a9c0ec9348d4f53c0b5eddbaff4876b22af291523fd7bb162a8ee90e,3,200,0x858323ababaab6995d2b8d768d95ae9ce0bafeefbc83bbcbf5581513a35c81a9,0x5049d8d9f770ce76897ac019d1971ab34e3db215c0ce647cafedfabb81390beb,False,DONE,COMPLETED,2.41,2.36
324,1,ethereum,2026-10-19T20:41:23,0xbe72462d374d4d1f89c64fb4f97ee6fa4b067711de7c775ff9a21a89176ec865,336,200,0x96d54526c03173ee1c8f37e28de93deeb59a37eabc0d8c6288c370be7f639db4,0xbe72462d374d4d1f89c64fb4f97ee6fa4b067711de7c775ff9a21a89176ec865,True,DONE,COMPLETED,0.02,0.02
325,2,ethereum,2026-10-19T20:46:11,0x8cac270c2ecea6b97850417748cb828ad9c4d831f4d0827f47e244cf0acef88b,299,200,0x96d54526c03173ee1c8f37e28de93deeb59a37eabc0d8c6288c370be7f639db4,0xbe72462d374d4d1f89c64fb4f97ee6fa4b067711de7c775ff9a21a89176ec865,False,DONE,COMPLETED,0.02,0.02


id lookup resolves to leg: {1: 6} (1 = earlier transaction, 2 = later)


### 5.2 Two-event transactions: which event the hash lookup returns, and where the other leg is

In [7]:
# 5.2 Two-event transactions: which event the hash lookup returns, what batchStatus holds, and whether the bridge parameter selects the leg.
two = ex[ex["reason_tags"].str.contains("two_event_tx")].sort_values(["chain", "tx_hash", "log_index"]).copy()
two["leg"] = two.groupby(["chain", "tx_hash"]).cumcount() + 1
two["own_id_in_batch_calls"] = [str(ids or "").lower().find(str(t).lower()) >= 0 for ids, t in zip(two["batch_call_transactionIds"], two["event_transactionId"])]
other_id = {(c, h, l): [t for l2, t in zip(g_["log_index"], g_["event_transactionId"]) if l2 != l][0] for (c, h), g_ in two.groupby(["chain", "tx_hash"]) for l in g_["log_index"]}
two["other_leg_id_in_response_text"] = [str(other_id[(c, h, int(l))]).lower() in json.dumps(edge_raw[("tx_hash", h, None)]["response"]).lower() for c, h, l in zip(two["chain"], two["tx_hash"], two["log_index"])]


def status_in_batch(row):
    "This event's status as batchStatus[].calls[] reports it (the call whose transactionId equals the event's)."
    ids = str(row["batch_call_transactionIds"] or "").lower().split("|")
    sts = str(row["batch_call_statuses"] or "").split("|")
    tid = str(row["event_transactionId"]).lower()
    return sts[ids.index(tid)] if tid in ids else None


two["own_status_in_batch_calls"] = two.apply(status_in_batch, axis=1)
print("hash lookups of the 22 rows (leg = order by log_index within the transaction; transaction_id_matches says whether the top-level record is this event; the batch columns come from batchStatus[].calls[]):")
display(two[["chain", "tx_hash", "leg", "log_index", "event_transactionId", "event_bridge", "event_integrator", "amount_usd_provided", "api_transactionId", "transaction_id_matches", "tool", "status", "substatus",
             "sending_amountUSD", "receiving_chainId", "receiving_amountUSD", "batch_n", "batch_calls_n", "own_id_in_batch_calls", "own_status_in_batch_calls", "other_leg_id_in_response_text", "batch_call_tools"]].round(2))
tx_level = two.groupby(["chain", "tx_hash"]).agg(bridges=("event_bridge", lambda s: "|".join(s)), returned_leg=("transaction_id_matches", lambda s: int(s.fillna(False).astype(bool).to_numpy().argmax()) + 1 if s.fillna(False).astype(bool).any() else None),
                                                 returned_tool=("tool", "first"), batch_n=("batch_n", "first"), batch_calls_n=("batch_calls_n", "first"), both_ids_in_batch_calls=("own_id_in_batch_calls", "all"),
                                                 other_leg_visible=("other_leg_id_in_response_text", "all"), batch_call_statuses=("batch_call_statuses", "first"), batch_success=("batch_success", "first"))
print("per transaction: which leg the top-level record is (1 = lower log_index), and whether batchStatus[].calls[] carries both events with a status each:")
display(tx_level)
print("returned leg by position:", tx_level["returned_leg"].value_counts(dropna=False).to_dict(), "; returned tool:", tx_level["returned_tool"].value_counts().to_dict(),
      "; transactions whose batchStatus carries both events:", int(tx_level["both_ids_in_batch_calls"].sum()), "of", len(tx_level), "; second-leg statuses from the batch:", two[two["leg"] == 2]["own_status_in_batch_calls"].value_counts(dropna=False).to_dict())
bp = edge_status[edge_status["lookup"] == "tx_hash_bridge"].copy()
bp["same_as_hash_lookup"] = bp["api_transactionId"].str.lower().to_numpy() == bp.set_index(["chain", "tx_hash"]).index.map(two.groupby(["chain", "tx_hash"])["api_transactionId"].first().str.lower()).to_numpy()
print("the bridge parameter (txHash plus bridge=<this event's bridge>): does the response change?")
display(bp[["chain", "tx_hash", "log_index", "event_bridge", "bridge_param", "http_status", "api_transactionId", "transaction_id_matches", "tool", "status", "substatus", "same_as_hash_lookup"]])
print("bridge-parameter lookups returning this event:", int(bp["transaction_id_matches"].fillna(False).astype(bool).sum()), "of", len(bp), "; identical to the plain hash lookup:", int(bp["same_as_hash_lookup"].sum()))

hash lookups of the 22 rows (leg = order by log_index within the transaction; transaction_id_matches says whether the top-level record is this event; the batch columns come from batchStatus[].calls[]):


,chain,tx_hash,leg,log_index,event_transactionId,event_bridge,event_integrator,amount_usd_provided,api_transactionId,transaction_id_matches,tool,status,substatus,sending_amountUSD,receiving_chainId,receiving_amountUSD,batch_n,batch_calls_n,own_id_in_batch_calls,own_status_in_batch_calls,other_leg_id_in_response_text,batch_call_tools
66,base,0x2890edb4603331e4c2fc9fea4a0f330637228c8f1e78950f284fc4576c75fa1d,1,247,0x664ab7f9233e562423f14f7eb3c6a86c30878256f4feec00baf9afa7f8fa729e,gasZipBridge,base-app,0.89,0x664ab7f9233e562423f14f7eb3c6a86c30878256f4feec00baf9afa7f8fa729e,True,gasZipBridge,DONE,COMPLETED,0.90,56,0.89,2,2,True,DONE/COMPLETED,True,gasZipBridge|mayan
67,base,0x2890edb4603331e4c2fc9fea4a0f330637228c8f1e78950f284fc4576c75fa1d,2,271,0xf4631bab03ec9937436087296d8c6ca6960e388155536d00e1417c4113fa9bdd,mayan,base-app,96.00,0x664ab7f9233e562423f14f7eb3c6a86c30878256f4feec00baf9afa7f8fa729e,False,gasZipBridge,DONE,COMPLETED,0.90,56,0.89,2,2,True,DONE/COMPLETED,True,gasZipBridge|mayan
55,base,0x28a2d2b3ce6b97d94646fce202fb8aef93b7845312e1e68fb268dcb0e7a8dae9,1,785,0xfc35543ffa183de35be95808bc1f6c9ab9069bff0d9ff8982c7c15edfcc74c79,near,base-app,5.54,0xfc35543ffa183de35be95808bc1f6c9ab9069bff0d9ff8982c7c15edfcc74c79,True,near,DONE,COMPLETED,5.61,1151111081099710,5.54,2,2,True,DONE/COMPLETED,True,near|mayan
56,base,0x28a2d2b3ce6b97d94646fce202fb8aef93b7845312e1e68fb268dcb0e7a8dae9,2,839,0x6a9be92dc99bf108a4f04b5184b49e76e6d22dbd84c362f223cd31d5394a54f8,mayan,base-app,15.07,0xfc35543ffa183de35be95808bc1f6c9ab9069bff0d9ff8982c7c15edfcc74c79,False,near,DONE,COMPLETED,5.61,1151111081099710,5.54,2,2,True,DONE/COMPLETED,True,near|mayan
76,base,0x3f091405413cfbe1332d118abb70b126a4133819e227c265e490b9a4d3140747,1,509,0x55c37f48b62959b47fde6bec564f28c762ac7f63f0bf64fd4bddb826abdef12f,mayan,base-app,239.72,0x55c37f48b62959b47fde6bec564f28c762ac7f63f0bf64fd4bddb826abdef12f,True,mayan,DONE,COMPLETED,250.91,1151111081099710,236.21,2,2,True,DONE/COMPLETED,True,mayan|gasZipBridge
77,base,0x3f091405413cfbe1332d118abb70b126a4133819e227c265e490b9a4d3140747,2,536,0x7a3678e4f384d544d13720d843e6ac98cad81dd23726a518aa87179e430f0e05,gasZipBridge,base-app,1.76,0x55c37f48b62959b47fde6bec564f28c762ac7f63f0bf64fd4bddb826abdef12f,False,mayan,DONE,COMPLETED,250.91,1151111081099710,236.21,2,2,True,DONE/COMPLETED,True,mayan|gasZipBridge
106,base,0x8d6b4a802561b45710e9f718082a7e8255ea43a679e7f66201b6f9d3be20a982,1,546,0x0b54fa2c6a21175c73e0338966e7d08e17e8d0482b8767a6fcd779a2f1fd2797,mayan,base-app,15.52,0x0b54fa2c6a21175c73e0338966e7d08e17e8d0482b8767a6fcd779a2f1fd2797,True,mayan,DONE,COMPLETED,16.15,137,43285.48,2,2,True,DONE/COMPLETED,True,mayan|mayan
107,base,0x8d6b4a802561b45710e9f718082a7e8255ea43a679e7f66201b6f9d3be20a982,2,570,0x2e685da79dc48d704cc86a1014b1aa984c6bd2aa4bb57ad1bd891eae908ac0af,mayan,base-app,9.40,0x0b54fa2c6a21175c73e0338966e7d08e17e8d0482b8767a6fcd779a2f1fd2797,False,mayan,DONE,COMPLETED,16.15,137,43285.48,2,2,True,DONE/COMPLETED,True,mayan|mayan
102,base,0xcc5348c107f825f334ba73313abb126fb296a9301087972855d29aea3bd4cc95,1,1412,0x6c939c84de84c74704e04d7c4882695687245309c36d4d2578dbeed44b64789b,mayan,base-app,10.48,0x6c939c84de84c74704e04d7c4882695687245309c36d4d2578dbeed44b64789b,True,mayan,DONE,COMPLETED,10.54,1151111081099710,10.47,2,2,True,DONE/COMPLETED,True,mayan|mayan
103,base,0xcc5348c107f825f334ba73313abb126fb296a9301087972855d29aea3bd4cc95,2,1438,0xa2eeb4dae124f0091c209649c37d15ede479521fda639f701ee89bf0659e8461,mayan,base-app,5.42,0x6c939c84de84c74704e04d7c4882695687245309c36d4d2578dbeed44b64789b,False,mayan,DONE,COMPLETED,10.54,1151111081099710,10.47,2,2,True,DONE/COMPLETED,True,mayan|mayan


per transaction: which leg the top-level record is (1 = lower log_index), and whether batchStatus[].calls[] carries both events with a status each:


bridges  returned_leg    returned_tool  batch_n  batch_calls_n  both_ids_in_batch_calls  \
chain    tx_hash                                                                                                                                                                               
base     0x2890edb4603331e4c2fc9fea4a0f330637228c8f1e78950f284fc4576c75fa1d               gasZipBridge|mayan             1     gasZipBridge        2              2                     True   
         0x28a2d2b3ce6b97d94646fce202fb8aef93b7845312e1e68fb268dcb0e7a8dae9                       near|mayan             1             near        2              2                     True   
         0x3f091405413cfbe1332d118abb70b126a4133819e227c265e490b9a4d3140747               mayan|gasZipBridge             1            mayan        2              2                     True   
         0x8d6b4a802561b45710e9f718082a7e8255ea43a679e7f66201b6f9d3be20a982                      mayan|mayan             1            mayan        2              2                     True   
         0xcc5348c107f825f334ba73313abb126fb296a9301087972855d29aea3bd4cc95                      mayan|mayan             1            mayan        2              2                     True   
bnb      0x34b516e155b17e02ab6034ae54b7da73cfec15b338a266d510a910efb2f5f0a8  relaydepository|relaydepository             1  relaydepository        2              2                     True   
         0x4d7057a7c52e2b0c097b94a048e6247635d2f82583709589dfa1dbbc6a685b28                    across|across             1           across        2              2                     True   
         0x8675d60f8a4e575140dd0cd5108a710d9f52cdb3e20b6fce031919da9d9708f1                    across|across             1           across        2              2                     True   
ethereum 0x2cb7afa8b00daeafd1ceffd7292fb24cb47d858ad40346a38039341a56d42d46     gasZipBridge|relaydepository             1     gasZipBridge        1              2                     True   
         0x4e0fec00b4e679bb59855f831cc78ac74efb64cad93c6801eb7ca7f0deb0c098                     across|mayan             1           across        2              2                     True   
         0xbd06ec4a055fe6abecbef782f11352211fada5462f76f7d591b8d51230542e8e          polymer|relaydepository             1          polymer     <NA>           <NA>                    False   

                                                                             other_leg_visible            batch_call_statuses batch_success  
chain    tx_hash                                                                                                                             
base     0x2890edb4603331e4c2fc9fea4a0f330637228c8f1e78950f284fc4576c75fa1d               True  DONE/COMPLETED|DONE/COMPLETED     True|True  
         0x28a2d2b3ce6b97d94646fce202fb8aef93b7845312e1e68fb268dcb0e7a8dae9               True  DONE/COMPLETED|DONE/COMPLETED     True|True  
         0x3f091405413cfbe1332d118abb70b126a4133819e227c265e490b9a4d3140747               True  DONE/COMPLETED|DONE/COMPLETED     True|True  
         0x8d6b4a802561b45710e9f718082a7e8255ea43a679e7f66201b6f9d3be20a982               True  DONE/COMPLETED|DONE/COMPLETED     True|True  
         0xcc5348c107f825f334ba73313abb126fb296a9301087972855d29aea3bd4cc95               True  DONE/COMPLETED|DONE/COMPLETED     True|True  
bnb      0x34b516e155b17e02ab6034ae54b7da73cfec15b338a266d510a910efb2f5f0a8               True  DONE/COMPLETED|DONE/COMPLETED     True|True  
         0x4d7057a7c52e2b0c097b94a048e6247635d2f82583709589dfa1dbbc6a685b28               True  DONE/COMPLETED|DONE/COMPLETED     True|True  
         0x8675d60f8a4e575140dd0cd5108a710d9f52cdb3e20b6fce031919da9d9708f1               True  DONE/COMPLETED|DONE/COMPLETED     True|True  
ethereum 0x2cb7afa8b00daeafd1ceffd7292fb24cb47d858ad40346a38039341a56d42d46               True  DONE/COMPLETED|DONE/COMPLETED          True  
         0x4e0fec00b4e679bb59855f831cc78ac

returned leg by position: {1: 11} ; returned tool: {'mayan': 3, 'across': 3, 'gasZipBridge': 2, 'near': 1, 'relaydepository': 1, 'polymer': 1} ; transactions whose batchStatus carries both events: 10 of 11 ; second-leg statuses from the batch: {'DONE/COMPLETED': 10, nan: 1}
the bridge parameter (txHash plus bridge=<this event's bridge>): does the response change?


,chain,tx_hash,log_index,event_bridge,bridge_param,http_status,api_transactionId,transaction_id_matches,tool,status,substatus,same_as_hash_lookup
294,base,0x28a2d2b3ce6b97d94646fce202fb8aef93b7845312e1e68fb268dcb0e7a8dae9,785,near,near,200,0xfc35543ffa183de35be95808bc1f6c9ab9069bff0d9ff8982c7c15edfcc74c79,True,near,DONE,COMPLETED,True
295,base,0x28a2d2b3ce6b97d94646fce202fb8aef93b7845312e1e68fb268dcb0e7a8dae9,839,mayan,mayan,200,0xfc35543ffa183de35be95808bc1f6c9ab9069bff0d9ff8982c7c15edfcc74c79,False,near,DONE,COMPLETED,True
296,base,0x2890edb4603331e4c2fc9fea4a0f330637228c8f1e78950f284fc4576c75fa1d,247,gasZipBridge,gasZipBridge,200,0x664ab7f9233e562423f14f7eb3c6a86c30878256f4feec00baf9afa7f8fa729e,True,gasZipBridge,DONE,COMPLETED,True
297,base,0x2890edb4603331e4c2fc9fea4a0f330637228c8f1e78950f284fc4576c75fa1d,271,mayan,mayan,200,0x664ab7f9233e562423f14f7eb3c6a86c30878256f4feec00baf9afa7f8fa729e,False,gasZipBridge,DONE,COMPLETED,True
298,base,0x3f091405413cfbe1332d118abb70b126a4133819e227c265e490b9a4d3140747,509,mayan,mayan,200,0x55c37f48b62959b47fde6bec564f28c762ac7f63f0bf64fd4bddb826abdef12f,True,mayan,DONE,COMPLETED,True
299,base,0x3f091405413cfbe1332d118abb70b126a4133819e227c265e490b9a4d3140747,536,gasZipBridge,gasZipBridge,200,0x55c37f48b62959b47fde6bec564f28c762ac7f63f0bf64fd4bddb826abdef12f,False,mayan,DONE,COMPLETED,True
300,base,0xcc5348c107f825f334ba73313abb126fb296a9301087972855d29aea3bd4cc95,1412,mayan,mayan,200,0x6c939c84de84c74704e04d7c4882695687245309c36d4d2578dbeed44b64789b,True,mayan,DONE,COMPLETED,True
301,base,0xcc5348c107f825f334ba73313abb126fb296a9301087972855d29aea3bd4cc95,1438,mayan,mayan,200,0x6c939c84de84c74704e04d7c4882695687245309c36d4d2578dbeed44b64789b,False,mayan,DONE,COMPLETED,True
302,base,0x8d6b4a802561b45710e9f718082a7e8255ea43a679e7f66201b6f9d3be20a982,546,mayan,mayan,200,0x0b54fa2c6a21175c73e0338966e7d08e17e8d0482b8767a6fcd779a2f1fd2797,True,mayan,DONE,COMPLETED,True
303,base,0x8d6b4a802561b45710e9f718082a7e8255ea43a679e7f66201b6f9d3be20a982,570,mayan,mayan,200,0x0b54fa2c6a21175c73e0338966e7d08e17e8d0482b8767a6fcd779a2f1fd2797,False,mayan,DONE,COMPLETED,True


bridge-parameter lookups returning this event: 11 of 22 ; identical to the plain hash lookup: 22


In [8]:
# The raw batchStatus of one two-event transaction, from the JSONL: one entry per user operation (index, sender, success, gas), each with `calls`, every call a full transfer record.
example = tx_level.index[0]
rec = edge_raw[("tx_hash", example[1], None)]
batch = rec["response"].get("batchStatus")
print(example, "batchStatus type:", type(batch).__name__, "entries:", len(batch) if isinstance(batch, list) else None, "| top-level keys:", sorted(rec["response"].keys()))
if isinstance(batch, list):
    for b in batch:
        print(f"entry index {b.get('index')}: sender {b.get('sender')}, success {b.get('success')}, gasAmountUSD {(b.get('gas') or {}).get('gasAmountUSD')}, userOpHash {str(b.get('userOpHash'))[:18]}…, keys {sorted(b.keys())}, calls {len(b.get('calls') or [])}")
        for c in b.get("calls") or []:
            s_, rcv = c.get("sending") or {}, c.get("receiving") or {}
            print("   call:", {"transactionId": c.get("transactionId"), "tool": c.get("tool"), "status": f"{c.get('status')}/{c.get('substatus')}", "metadata": c.get("metadata"),
                              "sending": (s_.get("chainId"), (s_.get("token") or {}).get("symbol"), s_.get("amountUSD")), "receiving": (rcv.get("chainId"), (rcv.get("token") or {}).get("symbol"), rcv.get("amountUSD")),
                              "toAddress": c.get("toAddress"), "feeCosts": [(f.get("name"), f.get("description"), f.get("amountUSD")) for f in (c.get("feeCosts") or [])], "keys": sorted(c.keys())})
print("event rows of that transaction:")
display(two[(two["chain"] == example[0]) & (two["tx_hash"] == example[1])][["log_index", "event_transactionId", "event_bridge", "event_integrator", "event_sendingAssetId", "event_destinationChainId", "amount_usd_provided"]])
print("every two-event transaction with a batchStatus: one line per user operation (sender = the account that signed the operation; tx_from in the data is the transaction's sender, the bundler):")
ops = []
for (c, h), g_ in two.groupby(["chain", "tx_hash"]):
    b = edge_raw[("tx_hash", h, None)]["response"].get("batchStatus")
    for e in (b or []):
        for call in e.get("calls") or []:
            ops.append({"chain": c, "tx_hash": h[:14] + "…", "tx_from": g_["event_tx_from"].iloc[0][:14] + "…", "op_index": e.get("index"), "op_sender": str(e.get("sender")).lower()[:14] + "…", "op_success": e.get("success"),
                        "call_transactionId": str(call.get("transactionId"))[:14] + "…", "call_tool": call.get("tool"), "call_status": f"{call.get('status')}/{call.get('substatus')}", "call_fromAddress": str(call.get("fromAddress"))[:14] + "…",
                        "call_integrator": (call.get("metadata") or {}).get("integrator"), "call_fee_names": "|".join(str(f.get("name")) for f in (call.get("feeCosts") or []))})
ops = pd.DataFrame(ops)
display(ops)
print("distinct operation senders per batched transaction:", ops.groupby(["chain", "tx_hash"])["op_sender"].nunique().value_counts().to_dict(), "; operations whose sender equals the data's tx_from:", int((ops["op_sender"] == ops["tx_from"]).sum()), "of", len(ops))

('base', '0x2890edb4603331e4c2fc9fea4a0f330637228c8f1e78950f284fc4576c75fa1d') batchStatus type: list entries: 2 | top-level keys: ['batchStatus', 'bridgeExplorerLink', 'fromAddress', 'lifiExplorerLink', 'metadata', 'quote', 'receiving', 'sending', 'status', 'substatus', 'substatusMessage', 'toAddress', 'tool', 'transactionId']
entry index 0: sender 0x583651cF25e4d051851749fD69c23EFfF32F01d2, success True, gasAmountUSD 0.0046, userOpHash 0x8a4ab0be14477cb7…, keys ['calls', 'gas', 'index', 'sender', 'success', 'userOpHash'], calls 1
   call: {'transactionId': '0x664ab7f9233e562423f14f7eb3c6a86c30878256f4feec00baf9afa7f8fa729e', 'tool': 'gasZipBridge', 'status': 'DONE/COMPLETED', 'metadata': {'integrator': 'base-app'}, 'sending': (8453, 'ETH', '0.8559'), 'receiving': (56, 'BNB', '0.9027'), 'toAddress': '0x583651cf25e4d051851749fd69c23efff32f01d2', 'feeCosts': [('LIFI Fee', 'Lifi Integrator Fee', '0.0086')], 'keys': ['bridgeExplorerLink', 'callIndex', 'feeCosts', 'fromAddress', 'lifiExplo

,log_index,event_transactionId,event_bridge,event_integrator,event_sendingAssetId,event_destinationChainId,amount_usd_provided
66,247,0x664ab7f9233e562423f14f7eb3c6a86c30878256f4feec00baf9afa7f8fa729e,gasZipBridge,base-app,0x0000000000000000000000000000000000000000,56,0.890469
67,271,0xf4631bab03ec9937436087296d8c6ca6960e388155536d00e1417c4113fa9bdd,mayan,base-app,0x0000000000000000000000000000000000000000,1151111081099710,95.995963


every two-event transaction with a batchStatus: one line per user operation (sender = the account that signed the operation; tx_from in the data is the transaction's sender, the bundler):


,chain,tx_hash,tx_from,op_index,op_sender,op_success,call_transactionId,call_tool,call_status,call_fromAddress,call_integrator,call_fee_names
0,base,0x2890edb46033…,0x1278c1e48e3c…,0,0x583651cf25e4…,True,0x664ab7f9233e…,gasZipBridge,DONE/COMPLETED,0x583651cf25e4…,base-app,LIFI Fee
1,base,0x2890edb46033…,0x1278c1e48e3c…,1,0x41417efd3188…,True,0xf4631bab03ec…,mayan,DONE/COMPLETED,0x41417efd3188…,base-app,LIFI Fee
2,base,0x28a2d2b3ce6b…,0xbdbebd58cc81…,0,0xe55dfc0a442e…,True,0xfc35543ffa18…,near,DONE/COMPLETED,0xe55dfc0a442e…,base-app,LIFI Fee
3,base,0x28a2d2b3ce6b…,0xbdbebd58cc81…,1,0xd26c5397aa32…,True,0x6a9be92dc99b…,mayan,DONE/COMPLETED,0xd26c5397aa32…,base-app,LIFI Fee
4,base,0x3f091405413c…,0x1278c1e48e3c…,1,0xbeb66e832a94…,True,0x55c37f48b629…,mayan,DONE/COMPLETED,0xbeb66e832a94…,base-app,LIFI Fee
5,base,0x3f091405413c…,0x1278c1e48e3c…,2,0x96ef0da02b8f…,True,0x7a3678e4f384…,gasZipBridge,DONE/COMPLETED,0x96ef0da02b8f…,base-app,LIFI Fee
6,base,0x8d6b4a802561…,0x4b5c33c3c7a3…,0,0xf2b46f14278d…,True,0x0b54fa2c6a21…,mayan,DONE/COMPLETED,0xf2b46f14278d…,base-app,LIFI Fee
7,base,0x8d6b4a802561…,0x4b5c33c3c7a3…,1,0x3336c6962bf8…,True,0x2e685da79dc4…,mayan,DONE/COMPLETED,0x3336c6962bf8…,base-app,LIFI Fee
8,base,0xcc5348c107f8…,0x54e2acab04c8…,1,0x80eff5f871f3…,True,0x6c939c84de84…,mayan,DONE/COMPLETED,0x80eff5f871f3…,base-app,LIFI Fee
9,base,0xcc5348c107f8…,0x54e2acab04c8…,2,0x2c553f12b249…,True,0xa2eeb4dae124…,mayan,DONE/COMPLETED,0x2c553f12b249…,base-app,LIFI Fee


distinct operation senders per batched transaction: {2: 9, 1: 1} ; operations whose sender equals the data's tx_from: 0 of 20


### 5.3 Transfers above $1M, and the 9,519,447.993443 USDC round trip

In [9]:
# 5.3 Transfers above $1M under the provided-feed column, and the 9,519,447.993443 USDC round trip.
big = ex[ex["reason_tags"].str.contains("above_1m_usd")].sort_values("amount_usd_provided", ascending=False).copy()
big["recv_over_send"] = (big["receiving_amountUSD"] / big["sending_amountUSD"]).round(4)
big["api_over_provided"] = (big["sending_amountUSD"] / big["amount_usd_provided"]).round(4)
display(big[["chain", "block_time", "event_bridge", "event_integrator", "event_destinationChainId", "sending_token_symbol", "amount_usd_provided", "sending_amountUSD", "api_over_provided", "status", "substatus",
             "receiving_chainId", "receiving_token_symbol", "receiving_amountUSD", "recv_over_send", "event_hasSourceSwaps", "transaction_id_matches", "reason_tags"]].round(2))
print(f"22 rows: completed {int(big['completed'].sum())}; sum provided USD {big['amount_usd_provided'].sum():,.2f}, API sending USD {big['sending_amountUSD'].sum():,.2f}, API receiving USD {big['receiving_amountUSD'].sum():,.2f}")
print("by bridge and integrator:", big.groupby(["event_bridge", "event_integrator"]).size().to_dict())
print(f"API sending USD over provided USD: min {big['api_over_provided'].min():.4f}, max {big['api_over_provided'].max():.4f}; receiving over sending: min {big['recv_over_send'].min():.4f}, max {big['recv_over_send'].max():.4f}; "
      f"sum receiving over sum sending {big['receiving_amountUSD'].sum() / big['sending_amountUSD'].sum():.4f}")
rt = ex[ex["reason_tags"].str.contains("round_trip_9519447")].sort_values("block_time")
print("the round trip, both legs:")
display(rt[["chain", "block_time", "tx_hash", "event_bridge", "event_integrator", "event_destinationChainId", "event_minAmount", "sending_amount", "sending_token_symbol", "sending_amountUSD", "status", "substatus", "tool",
            "receiving_chainId", "receiving_token_symbol", "receiving_amount", "receiving_amountUSD", "receiving_txHash", "receiving_timestamp", "fromAddress", "toAddress", "event_tx_from"]].T)
a, b = rt.iloc[0], rt.iloc[1]
print(f"leg 1 received {int(a['receiving_amount']) / 1e6 if pd.notna(a['receiving_amount']) else None} on chain {a['receiving_chainId']}; leg 2 sent {int(b['sending_amount']) / 1e6 if pd.notna(b['sending_amount']) else None} from chain {b['sending_chainId']}; "
      f"leg 2 sent minus leg 1 received: {(int(b['sending_amount']) - int(a['receiving_amount'])) / 1e6 if pd.notna(a['receiving_amount']) and pd.notna(b['sending_amount']) else None} USDC; "
      f"same sender address both legs: {a['fromAddress'] == b['fromAddress']}; leg 1 toAddress equals leg 2 fromAddress: {str(a['toAddress']).lower() == str(b['fromAddress']).lower()}")

,chain,block_time,event_bridge,event_integrator,event_destinationChainId,sending_token_symbol,amount_usd_provided,sending_amountUSD,api_over_provided,status,substatus,receiving_chainId,receiving_token_symbol,receiving_amountUSD,recv_over_send,event_hasSourceSwaps,transaction_id_matches,reason_tags
249,ethereum,2026-10-18T16:40:35,polymerStandard,_arbitrum,42161,USDC,10002421.67,10009262.01,1.00,DONE,COMPLETED,42161,USDC,9997725.31,1.00,False,True,above_1m_usd
248,ethereum,2026-10-18T16:19:23,polymerStandard,_arbitrum,42161,USDC,10002421.67,10009262.01,1.00,DONE,COMPLETED,42161,USDC,9996311.19,1.00,False,True,above_1m_usd
222,ethereum,2026-10-16T13:39:59,polymerStandard,Pharos-Bridge,1672,USDC,10000569.17,9995861.10,1.00,DONE,COMPLETED,1672,USDC,10002129.18,1.00,False,True,above_1m_usd
250,ethereum,2026-10-18T17:01:59,polymerStandard,_arbitrum,42161,USDC,9521501.02,9528264.91,1.00,DONE,COMPLETED,42161,USDC,9525900.11,1.00,False,True,above_1m_usd|round_trip_9519447
44,arbitrum,2026-10-19T17:57:04,polymerStandard,_arbitrum,1,USDC,9520869.56,9507101.07,1.00,DONE,COMPLETED,1,USDC,9513150.96,1.00,False,True,above_1m_usd|round_trip_9519447
223,ethereum,2026-10-16T13:54:59,polymerStandard,Pharos-Bridge,1672,USDC,6505356.54,6502293.95,1.00,DONE,COMPLETED,1672,USDC,6506371.33,1.00,False,True,above_1m_usd
12,arbitrum,2026-10-15T00:04:58,glacis,jumper.exchange,1,PYUSD,2013311.27,2012815.53,1.00,DONE,COMPLETED,1,PYUSD,2012996.53,1.00,False,True,above_1m_usd
170,ethereum,2026-10-14T00:23:47,polymerStandard,_binancewallet,1672,USDC,2000378.17,1999334.64,1.00,DONE,COMPLETED,1672,USDC,1995938.42,1.00,False,True,above_1m_usd
13,arbitrum,2026-10-15T00:53:01,glacis,USDai,1,USDai,1655006.40,1652136.79,1.00,DONE,COMPLETED,1,PYUSD,1650394.91,1.00,True,True,above_1m_usd
226,ethereum,2026-10-16T15:52:23,glacis,safe-app,4663,USDG,1517757.14,1518048.66,1.00,DONE,COMPLETED,4663,USDG,1516177.68,1.00,True,True,above_1m_usd


22 rows: completed 22; sum provided USD 76,115,424.09, API sending USD 76,106,990.87, API receiving USD 76,093,712.24
by bridge and integrator: {('glacis', 'USDai'): 3, ('glacis', '_arbitrum'): 1, ('glacis', 'jumper.exchange'): 1, ('glacis', 'oku.trade'): 3, ('glacis', 'safe-app'): 1, ('polymerStandard', 'Pharos-Bridge'): 2, ('polymerStandard', '_arbitrum'): 4, ('polymerStandard', '_binancewallet'): 3, ('polymerStandard', 'jumper.exchange'): 1, ('polymerStandard', 'jumperadvanced'): 1, ('relaydepository', 'mbridge-stable'): 2}
API sending USD over provided USD: min 0.9948, max 1.0016; receiving over sending: min 0.9953, max 1.0062; sum receiving over sum sending 0.9998
the round trip, both legs:


,250,44
chain,ethereum,arbitrum
block_time,2026-10-18T17:01:59,2026-10-19T17:57:04
tx_hash,0x08f630e8c0afd0d8f9a9af9af6c82d2e908713534fecc858ba8fe947fbf8bb26,0x436b3698f1db02d8e5c9391310f19e9abf87e336f96bd185b3e5ed52dfb162a9
event_bridge,polymerStandard,polymerStandard
event_integrator,_arbitrum,_arbitrum
event_destinationChainId,42161,1
event_minAmount,9519447993443,9519447993443
sending_amount,9519447993443,9519447993443
sending_token_symbol,USDC,USDC
sending_amountUSD,9528264.9128,9507101.0666


leg 1 received 9519447.993443 on chain 42161; leg 2 sent 9519447.993443 from chain 42161; leg 2 sent minus leg 1 received: 0.0 USDC; same sender address both legs: True; leg 1 toAddress equals leg 2 fromAddress: True


### 5.4 Destinations LI.FI's chain list does not carry: the receiving chain and token

In [10]:
# 5.4 Destinations LI.FI's chain list does not carry: the receiving chain id and token the status reports.
unk = ex[ex["reason_tags"].str.contains("unknown_dest")].copy()
print("event destination id × receiving chain id reported by the API (rows):")
display(pd.crosstab(unk["event_destinationChainId"], unk["receiving_chainId"].astype("Int64").astype(str).replace("<NA>", "(none)"), margins=True))
print("per destination id: statuses, bridges, receiving tokens, the API's toAddress format:")
unk["toAddress_len"] = unk["toAddress"].str.len()
unk["toAddress_prefix"] = unk["toAddress"].str[:2]
display(unk.groupby("event_destinationChainId").agg(rows=("tx_hash", "size"), completed=("completed", "sum"), refunded=("substatus", lambda s: int((s == "REFUNDED").sum())),
                                                   bridges=("event_bridge", lambda s: "|".join(f"{k}:{v}" for k, v in s.value_counts().items())),
                                                   receiving_chain_ids=("receiving_chainId", lambda s: "|".join(f"{k}:{v}" for k, v in s.value_counts(dropna=False).items())),
                                                   receiving_tokens=("receiving_token_symbol", lambda s: "|".join(f"{k}:{v}" for k, v in s.value_counts(dropna=False).items())),
                                                   sending_usd=("sending_amountUSD", "sum"), receiving_usd=("receiving_amountUSD", "sum"),
                                                   toAddress_len=("toAddress_len", lambda s: "|".join(map(str, sorted(set(s.dropna().astype(int)))))), toAddress_prefix=("toAddress_prefix", lambda s: "|".join(sorted(set(s.dropna()))))).round(2))
print("rows to the five smaller ids:")
display(unk[unk["event_destinationChainId"] != UNKNOWN_BIG][["chain", "block_time", "event_bridge", "event_integrator", "event_destinationChainId", "event_receiver", "toAddress", "status", "substatus", "receiving_chainId", "receiving_token_symbol", "sending_amountUSD", "receiving_amountUSD"]].round(2))
chains_ref = pd.read_csv(REF / "lifi_api_chains.csv")
print("receiving chain ids seen above that the chain list names:", chains_ref[chains_ref["id"].isin(unk["receiving_chainId"].dropna().astype(int).unique())][["id", "key", "name", "chainType"]].to_dict("records"))
print("example toAddress per destination id (first row each):", unk.groupby("event_destinationChainId")["toAddress"].first().to_dict())

event destination id × receiving chain id reported by the API (rows):


receiving_chainId,1201081091099710,204,232,60808,728126428,81457,All
event_destinationChainId,,,,,,,
204,0,11,0,0,0,0,11
232,0,0,3,0,0,0,3
60808,0,0,0,4,0,0,4
81457,0,0,0,0,0,12,12
1201081091099710,4,0,0,0,0,0,4
1885080386571452,0,0,0,0,50,0,50
All,4,11,3,4,50,12,84


per destination id: statuses, bridges, receiving tokens, the API's toAddress format:


,rows,completed,refunded,bridges,receiving_chain_ids,receiving_tokens,sending_usd,receiving_usd,toAddress_len,toAddress_prefix
event_destinationChainId,,,,,,,,,,
204,11,11,0,symbiosis:6|gasZipBridge:4|layerswap:1,204:11,BNB:10|ETH:1,59.11,50.02,42,0x
232,3,3,0,gasZipBridge:3,232:3,GHO:3,7.76,7.67,42,0x
60808,4,4,0,layerswap:3|gasZipBridge:1,60808:4,ETH:4,547.27,534.83,42,0x
81457,12,12,0,layerswap:9|relaydepository:3,81457:12,ETH:11|USDB:1,666.76,659.19,42,0x
1201081091099710,4,4,0,polymer:3|near:1,1201081091099710:4,USDC:3|XLM:1,1231.06,1229.22,56,GA|GC
1885080386571452,50,50,0,near:33|symbiosis:9|layerswap:8,728126428:50,USDT:41|TRX:9,530460.56,528198.48,34,TA|TB|TC|TD|TE|TF|TG|TH|TJ|TK|TL|TM|TN|TP|TQ|TR|TS|TU|TV|TW|TX|TZ


rows to the five smaller ids:


,chain,block_time,event_bridge,event_integrator,event_destinationChainId,event_receiver,toAddress,status,substatus,receiving_chainId,receiving_token_symbol,sending_amountUSD,receiving_amountUSD
3,arbitrum,2026-10-14T06:04:04,gasZipBridge,lifi-api,60808,0xe3e22e7f170b4ee58d8824d8ebbf34bfa87289e2,0xe3e22e7f170b4ee58d8824d8ebbf34bfa87289e2,DONE,COMPLETED,60808,ETH,0.25,0.24
8,arbitrum,2026-10-14T10:07:28,layerswap,jumper.exchange,81457,0x44f8b4df7252023bbbe334ceb59e4acf93b4353a,0x44f8b4df7252023bbbe334ceb59e4acf93b4353a,DONE,COMPLETED,81457,ETH,0.75,0.74
21,arbitrum,2026-10-16T14:58:11,polymer,jumper.exchange,1201081091099710,0x11f111f111f111f111f111f111f111f111f111f1,GCMWJP2MMGBJHNQ2IP7NFEX6HGQFPNXV5YA27DCON4DJZUPKHPBFC2OP,DONE,COMPLETED,1201081091099710,USDC,6.00,5.94
51,base,2026-10-13T04:31:03,layerswap,jumper.exchange,81457,0xc8de67cac7eda919242417002123a9889eba9b1d,0xc8de67cac7eda919242417002123a9889eba9b1d,DONE,COMPLETED,81457,ETH,115.39,115.04
53,base,2026-10-13T12:11:31,layerswap,rabbykmsv2,81457,0x1343c26670625503b33ac805b81e94f2e0bc8c2e,0x1343c26670625503b33ac805b81e94f2e0bc8c2e,DONE,COMPLETED,81457,ETH,0.08,0.06
57,base,2026-10-14T01:28:35,layerswap,jumper.exchange,81457,0x27b7ed99367f5aa53dfa749a4fc7b0af078ba9fe,0x27b7ed99367f5aa53dfa749a4fc7b0af078ba9fe,DONE,COMPLETED,81457,ETH,0.13,0.12
61,base,2026-10-14T06:59:03,layerswap,jumper.exchange,81457,0xddf93172efb04fb7aff97b962ddd1c17180f1687,0xddf93172efb04fb7aff97b962ddd1c17180f1687,DONE,COMPLETED,81457,ETH,27.54,27.53
62,base,2026-10-14T07:56:57,layerswap,jumper.exchange,60808,0x102f479312f69157df8b804905a20fe5025881a5,0x102f479312f69157df8b804905a20fe5025881a5,DONE,COMPLETED,60808,ETH,0.25,0.19
63,base,2026-10-14T08:10:27,layerswap,rabbykmsv2,81457,0x1343c26670625503b33ac805b81e94f2e0bc8c2e,0x1343c26670625503b33ac805b81e94f2e0bc8c2e,DONE,COMPLETED,81457,ETH,0.08,0.06
64,base,2026-10-14T08:10:33,gasZipBridge,hey,232,0xe7e8b2e4c0b8722eb180cd67029fb4cb3654cc9c,0xe7e8b2e4c0b8722eb180cd67029fb4cb3654cc9c,DONE,COMPLETED,232,GHO,2.52,2.44


receiving chain ids seen above that the chain list names: [{'id': 728126428, 'key': 'trn', 'name': 'Tron', 'chainType': 'TVM'}]
example toAddress per destination id (first row each): {np.int64(204): '0xa9798b1b45902fed30cf56279170f05c8dc15841', np.int64(232): '0xe7e8b2e4c0b8722eb180cd67029fb4cb3654cc9c', np.int64(60808): '0xe3e22e7f170b4ee58d8824d8ebbf34bfa87289e2', np.int64(81457): '0x44f8b4df7252023bbbe334ceb59e4acf93b4353a', np.int64(1201081091099710): 'GCMWJP2MMGBJHNQ2IP7NFEX6HGQFPNXV5YA27DCON4DJZUPKHPBFC2OP', np.int64(1885080386571452): 'TCNN7yvs56YQ7v1PtMk5394kyZ67xnGkbR'}


### 5.5 Sentinel-receiver rows to non-EVM chains: receiving chain, address and completion by type

In [11]:
# 5.5 Sentinel-receiver rows to non-EVM chains: receiving chain, address format and completion by chain type.
sen = ex[ex["reason_tags"].str.contains("sentinel_")].copy()
sen["kind"] = sen["reason_tags"].str.extract(r"(sentinel_\w+)")
sen["toAddress_len"] = sen["toAddress"].str.len()
sen["toAddress_is_sentinel"] = sen["toAddress"].str.lower() == SENTINEL_RECEIVER
sen["toAddress_hex40"] = sen["toAddress"].str.match(r"^0x[0-9a-fA-F]{40}$").fillna(False)
display(sen.groupby("kind").agg(rows=("tx_hash", "size"), completed=("completed", "sum"), refunded=("substatus", lambda s: int((s == "REFUNDED").sum())), not_done=("status", lambda s: int((s != "DONE").sum())),
                                dest_ids=("event_destinationChainId", lambda s: "|".join(map(str, sorted(set(s))))), receiving_chain_ids=("receiving_chainId", lambda s: "|".join(f"{k}:{v}" for k, v in s.value_counts(dropna=False).items())),
                                dest_chain_matches=("dest_chain_matches", lambda s: int(s.fillna(False).astype(bool).sum())),
                                receiving_tokens=("receiving_token_symbol", lambda s: "|".join(f"{k}:{v}" for k, v in s.value_counts(dropna=False).items())),
                                toAddress_sentinel=("toAddress_is_sentinel", "sum"), toAddress_hex40=("toAddress_hex40", "sum"), toAddress_lens=("toAddress_len", lambda s: "|".join(map(str, sorted(set(s.dropna().astype(int)))))),
                                bridges=("event_bridge", lambda s: "|".join(f"{k}:{v}" for k, v in s.value_counts().items())), sending_usd=("sending_amountUSD", "sum"), receiving_usd=("receiving_amountUSD", "sum")).round(2))
print("example toAddress per kind (first two rows each):")
for kind, g_ in sen.groupby("kind"):
    print(f"  {kind}: {g_['toAddress'].head(2).tolist()}")
sen["addr_class"] = sen["toAddress"].apply(lambda a: "0x" if str(a).startswith("0x") else ("bc1" if str(a).startswith("bc1") else str(a)[0]))
print("toAddress format by kind, (first characters, length) -> rows; '0x' is a 20-byte EVM address at length 42 or a Sui address at length 66, 'bc1' Bitcoin bech32, '1' or '3' Bitcoin legacy or P2SH, any other character the first of a base58 Solana address:")
for kind, g_ in sen.groupby("kind"):
    print(f"  {kind}: {g_.groupby(['addr_class', 'toAddress_len']).size().to_dict()}")
print("rows to Solana or Bitcoin whose toAddress is a 0x address:")
display(sen[sen["kind"].isin(["sentinel_solana", "sentinel_utxo"]) & (sen["addr_class"] == "0x")][["kind", "chain", "event_bridge", "event_integrator", "status", "substatus", "receiving_chainId", "toAddress"]])
print("rows not completed:")
display(sen[~sen["completed"]][["kind", "chain", "block_time", "event_bridge", "event_integrator", "event_destinationChainId", "status", "substatus", "substatusMessage", "receiving_chainId", "sending_amountUSD", "receiving_amountUSD"]].round(2))
print("receiving over sending USD on completed sentinel rows, by kind:")
sen["recv_over_send"] = sen["receiving_amountUSD"] / sen["sending_amountUSD"]
display(sen[sen["completed"]].groupby("kind")["recv_over_send"].describe(percentiles=[0.1, 0.5, 0.9]).round(4))

,rows,completed,refunded,not_done,dest_ids,receiving_chain_ids,dest_chain_matches,receiving_tokens,toAddress_sentinel,toAddress_hex40,toAddress_lens,bridges,sending_usd,receiving_usd
kind,,,,,,,,,,,,,,
sentinel_solana,50,48,2,0,1151111081099710,1151111081099710:48|8453:1|56:1,48,SOL:29|USDC:11|USDT:3|BAGSPAY:1|arc:1|MASK:1|LOS:1|MOBY:1|BOOM:1|A...,0,1,42|43|44,mayan:27|relaydepository:11|near:8|lifiIntents:2|polymerStandard:2,30325.97,50698.78
sentinel_sui,50,47,0,0,9270000000000000,9270000000000000:50,50,USDC:26|SUI:17|US:3|WAL:3|WETH:1,0,0,66,mayanMCTP:50,38445.43,37369.39
sentinel_tron,2,2,0,0,728126428,728126428:2,2,USDT:2,0,2,42,lifiIntents:2,18.97,10.91
sentinel_utxo,50,50,0,0,20000000000001,20000000000001:50,50,BTC:50,0,0,34|42|62,layerswap:35|relaydepository:10|near:5,56430.76,55662.30


example toAddress per kind (first two rows each):
  sentinel_solana: ['ELnJFZSCYyWSuCYyxBoaGKfUiiqvuAPYMnRGY9qSYRFJ', 'E3RzdGTu9JdeUZcB4NikV4W1txEGPHEXFHbC7gVNxE6Z']
  sentinel_sui: ['0x6d970cadda4d369d679d183b1555a6901f88b2f25c7bd67c01991025cdd3723f', '0x7081da32f0966a4c1b8298481531d1535986db221fb1648c93f2cfcce1cc80fc']
  sentinel_tron: ['0xc63756a0dbd63cf54e694cbac072d22bddb8df5c', '0xc63756a0dbd63cf54e694cbac072d22bddb8df5c']
  sentinel_utxo: ['bc1p9uutgqnsspe94g6kmduf86v0pyy22ekx9r42wy33vfzrgshvqrfqcjwlj3', 'bc1q8ry8a890lw39fwg2hs08u06cttr3dajp4uck4a']
toAddress format by kind, (first characters, length) -> rows; '0x' is a 20-byte EVM address at length 42 or a Sui address at length 66, 'bc1' Bitcoin bech32, '1' or '3' Bitcoin legacy or P2SH, any other character the first of a base58 Solana address:
  sentinel_solana: {('0x', 42): 1, ('2', 44): 5, ('3', 44): 4, ('4', 44): 3, ('5', 44): 2, ('6', 44): 2, ('7', 44): 5, ('8', 44): 2, ('9', 44): 2, ('A', 44): 4, ('B', 44): 2, ('C', 44): 

,kind,chain,event_bridge,event_integrator,status,substatus,receiving_chainId,toAddress
144,sentinel_solana,bnb,lifiIntents,debot,DONE,REFUNDED,56,0xe79d52784e0223a3f4e345650a71e285bcac1965


rows not completed:


,kind,chain,block_time,event_bridge,event_integrator,event_destinationChainId,status,substatus,substatusMessage,receiving_chainId,sending_amountUSD,receiving_amountUSD
34,sentinel_sui,arbitrum,2026-10-18T21:46:39,mayanMCTP,jumper.exchange,9270000000000000,DONE,PARTIAL,Some of the received tokens are not the requested destination tokens.,9270000000000000,44.66,42.90
40,sentinel_sui,arbitrum,2026-10-19T14:26:37,mayanMCTP,jumper.exchange,9270000000000000,DONE,PARTIAL,Some of the received tokens are not the requested destination tokens.,9270000000000000,202.87,202.16
84,sentinel_solana,base,2026-10-16T22:40:29,mayan,base-app,1151111081099710,DONE,REFUNDED,The tokens were refunded to the user.,8453,3752.22,3608.01
144,sentinel_solana,bnb,2026-10-19T13:37:58,lifiIntents,debot,1151111081099710,DONE,REFUNDED,The tokens were refunded to the user.,56,20.94,20.89
188,sentinel_sui,ethereum,2026-10-14T14:25:59,mayanMCTP,tangem,9270000000000000,DONE,PARTIAL,Some of the received tokens are not the requested destination tokens.,9270000000000000,111.14,104.68


receiving over sending USD on completed sentinel rows, by kind:


,count,mean,std,min,10%,50%,90%,max
kind,,,,,,,,
sentinel_solana,48.0,4.6256,25.1747,0.8225,0.9656,0.9939,1.0110,175.4068
sentinel_sui,47.0,0.9375,0.1080,0.4562,0.8167,0.9775,1.0007,1.0038
sentinel_tron,2.0,0.5743,0.0274,0.5549,0.5588,0.5743,0.5898,0.5937
sentinel_utxo,50.0,0.9813,0.0180,0.9199,0.9554,0.9867,0.9986,1.0016


### 5.6 The two empty-integrator rows

In [12]:
# 5.6 The two empty-integrator rows.
emp = ex[ex["reason_tags"].str.contains("empty_integrator")]
display(emp[["chain", "block_time", "tx_hash", "event_bridge", "event_integrator", "event_destinationChainId", "event_tx_from", "amount_usd_provided", "status", "substatus", "tool", "metadata_integrator", "quote_integrator",
             "sending_amountUSD", "receiving_chainId", "receiving_amountUSD", "quote_fee_names", "quote_fee_lifi_usd", "fee_split_recipients"]].T)
emp_raw = [edge_raw[("tx_hash", h, None)]["response"] for h in emp["tx_hash"]]
print("provided USD of the two rows:", [round(v, 2) for v in emp["amount_usd_provided"]], "| API sending USD:", [round(v, 2) for v in emp["sending_amountUSD"]], "| LIFI-named fee USD:", list(emp["quote_fee_lifi_usd"]))
print("the quote of each row: fromAmountUSD, toAmountUSD, fee lines total, LIFI-named, other:")
display(emp[["tx_hash", "quote_fromAmountUSD", "quote_toAmountUSD", "quote_fee_usd", "quote_fee_lifi_usd", "quote_fee_other_usd", "quote_fee_names", "receiving_amountUSD"]].round(4))
for h, r in zip(emp["tx_hash"], emp_raw):
    est = (r.get("quote") or {}).get("estimate") or {}
    print(f"  {h[:14]}…: toAmountUSD {est.get('toAmountUSD')}, toAmount {est.get('toAmount')}, fee lines {[(f.get('name'), f.get('amountUSD'), f.get('included')) for f in est.get('feeCosts') or []]}")
print("the event's integrator string in the data, repr:", [repr(v) for v in tf.loc[tf["tx_hash"].isin(emp["tx_hash"]), "integrator"]], "(the CSV shows an empty cell as NaN)")
print("raw response metadata:", [r.get("metadata") for r in emp_raw], "| quote.integrator repr:", [repr((r.get("quote") or {}).get("integrator")) for r in emp_raw], "| quote present:", [bool(r.get("quote")) for r in emp_raw])

,280,281
chain,polygon,polygon
block_time,2026-10-13T07:37:23,2026-10-13T08:21:23
tx_hash,0x2b4889d16a4080bbc154efd84a8986e481b5176e004d3ba7e2ae7a8ffa29c00c,0x7b880738068c5d168b86ba551fb63a12b8a9c17276bc676c62e7d1de78a92226
event_bridge,across,across
event_integrator,NaN,NaN
event_destinationChainId,42161,42161
event_tx_from,0xc066ac5d385419b1a8c43a0e146fa439837a8b8c,0xb01caea8c6c47bbf4f4b4c5080ca642043359c2e
amount_usd_provided,1.062586,2.027363
status,DONE,DONE
substatus,COMPLETED,COMPLETED


provided USD of the two rows: [1.06, 2.03] | API sending USD: [1.06, 2.03] | LIFI-named fee USD: [0.0027, 0.0051]
the quote of each row: fromAmountUSD, toAmountUSD, fee lines total, LIFI-named, other:


,tx_hash,quote_fromAmountUSD,quote_toAmountUSD,quote_fee_usd,quote_fee_lifi_usd,quote_fee_other_usd,quote_fee_names,receiving_amountUSD
280,0x2b4889d16a4080bbc154efd84a8986e481b5176e004d3ba7e2ae7a8ffa29c00c,1.0648,0.0,0.0328,0.0027,0.0301,LIFI Fixed Fee|Relayer fee|Relayer gas fee,0.0038
281,0x7b880738068c5d168b86ba551fb63a12b8a9c17276bc676c62e7d1de78a92226,2.0310,0.0,0.0355,0.0051,0.0304,LIFI Fixed Fee|Relayer fee|Relayer gas fee,0.0038


  0x2b4889d16a40…: toAmountUSD 0.0000, toAmount 0, fee lines [('LIFI Fixed Fee', '0.0027', True), ('Relayer fee', '0.0001', True), ('Relayer gas fee', '0.0300', True)]
  0x7b880738068c…: toAmountUSD 0.0000, toAmount 0, fee lines [('LIFI Fixed Fee', '0.0051', True), ('Relayer fee', '0.0002', True), ('Relayer gas fee', '0.0302', True)]
the event's integrator string in the data, repr: ["''", "''"] (the CSV shows an empty cell as NaN)
raw response metadata: [{'integrator': ''}, {'integrator': ''}] | quote.integrator repr: ['None', 'None'] | quote present: [True, True]


## 6. Fetch: 2,000 further events drawn uniformly

Eligible rows: the 124,858 minus the 600 already sampled (01 §5) and minus the edge-case rows of §3. `DataFrame.sample(n=2000, random_state=20261029)` over the eligible rows in file order, so each chain's count is proportional to its activity. The snapshot `lifi_api_transfer_status_sample_2600.csv` holds these 2,000 (`sample_design = uniform_2000`) and the 600 re-flattened from their raw JSONL (`sample_design = equal_per_chain_600`, their original fetch times), so the two designs can be analysed apart or weighted. The 600 files stay as they are; the 2600 files supersede them for analysis.

In [13]:
S2600_CSV, S2600_JSONL = "lifi_api_transfer_status_sample_2600.csv", REF / "lifi_api_transfer_status_sample_2600_raw.jsonl"
KEDGE = set(edge["key"])
eligible = tf[~tf["key"].isin(K600 | KEDGE)]
sample2000 = eligible.sample(n=2000, random_state=SEED).sort_values(["chain", "block_time", "log_index"], kind="stable")
sample2000 = sample2000.assign(sample_design="uniform_2000")
events600 = tf[tf["key"].isin(K600)].sort_values(["chain", "block_time", "log_index"], kind="stable").assign(sample_design="equal_per_chain_600")
events2600 = pd.concat([sample2000, events600], ignore_index=True)
assert len(eligible) == 124_858 - len(K600 | KEDGE) and len(sample2000) == 2000 and sample2000["tx_hash"].is_unique and len(events2600) == 2600
per_chain = pd.DataFrame({"rows_in_data": tf["chain"].value_counts(), "sampled_2000": sample2000["chain"].value_counts()})
per_chain["row_share"] = (per_chain["rows_in_data"] / len(tf)).round(4)
per_chain["expected_proportional"] = (per_chain["row_share"] * 2000).round(1)
per_chain["sample_share"] = (per_chain["sampled_2000"] / 2000).round(4)
print(f"eligible rows {len(eligible):,} = 124,858 − {len(K600 | KEDGE)} (600 sampled ∪ {len(edge)} edge rows, {len(K600 & KEDGE)} in both); sampled {len(sample2000)} with seed {SEED}")
display(per_chain.sort_values("rows_in_data", ascending=False))
print("sampled 2,000 by bridge (top 12):", sample2000["bridge"].value_counts().head(12).to_dict())


def fetch_sample_2600():
    if not S2600_JSONL.exists():
        with S2600_JSONL.open("a") as fh:
            for key, r in sorted(s600_raw.items()):
                rec = dict(r)
                rec.update({"lookup": "tx_hash", "lookup_value": r["tx_hash"], "bridge_param": None, "request_params": {"txHash": r["tx_hash"]}, "sample_design": "equal_per_chain_600"})
                fh.write(json.dumps(rec) + "\n")
        print(f"seeded {S2600_JSONL.name} with the 600 raw records of lifi_api_transfer_status_sample_600_raw.jsonl")
    calls = [{"lookup": "tx_hash", "chain": ev.chain, "lookup_value": ev.tx_hash, "bridge_param": None, "params": {"txHash": ev.tx_hash},
              "log_index": int(ev.log_index), "event_transactionId": ev.transaction_id} for ev in sample2000.itertuples(index=False)]
    done = fetch_calls(calls, S2600_JSONL, label="sample 2,000",
                       extra=lambda c: {"tx_hash": c["lookup_value"], "log_index": c["log_index"], "event_transactionId": c["event_transactionId"], "sample_design": "uniform_2000"})
    rows = []
    for ev in events2600.to_dict("records"):
        row = flatten_status(ev, done[("tx_hash", ev["tx_hash"], None)])
        row.update({"sample_design": ev["sample_design"], "lookup": "tx_hash", "lookup_value": ev["tx_hash"]})
        rows.append(row)
    return pd.DataFrame(rows)


def s2600_note():
    recs = jsonl_read(S2600_JSONL)
    designs, statuses = Counter(r.get("sample_design") for r in recs), Counter(r["http_status"] for r in recs)
    new = [r for r in recs if r.get("sample_design") == "uniform_2000"]
    return (f"2,000 events drawn uniformly from the 124,858 rows minus the 600 of lifi_api_transfer_status_sample_600.csv and minus the {len(edge)} edge-case rows, "
            f"DataFrame.sample(n=2000, random_state={SEED}), per chain {sample2000['chain'].value_counts().to_dict()}; {fetch_span(new)}; plus the 600 of the earlier snapshot (01 §5, fetched 2026-10-28 14:48 to 16:03 UTC) "
            f"re-flattened from their raw JSONL with sample_design = equal_per_chain_600 and their original fetch times; records by design {dict(designs)}; HTTP statuses of all 2,600 records {dict(statuses)}; "
            f"raw responses in lifi_api_transfer_status_sample_2600_raw.jsonl")


s2600 = snapshot_csv(S2600_CSV, fetch_sample_2600, "03 §6", STATUS_URL, note=s2600_note,
                     fetched_at=lambda: min(r["fetched_at_utc"] for r in jsonl_read(S2600_JSONL) if r.get("sample_design") == "uniform_2000"))
s2600 = read_status_csv(S2600_CSV)
manifest = manifest_read()
if S2600_JSONL.name not in manifest:
    manifest_write(S2600_JSONL.name, STATUS_URL, sum(1 for _ in S2600_JSONL.open()), "03 §6", manifest[S2600_CSV]["fetched_at_utc"],
                   note="raw JSON responses, one line per event (2,000 new plus the 600 copied from lifi_api_transfer_status_sample_600_raw.jsonl), with sample_design, the event keys, request parameters (no key) and HTTP status")
print(f"{S2600_CSV}: {s2600.shape}; by design {s2600['sample_design'].value_counts().to_dict()}; HTTP statuses {s2600['http_status'].value_counts().to_dict()}")
print("status/substatus by design:")
display(pd.crosstab([s2600["sample_design"], s2600["status"].fillna("(none)")], s2600["substatus"].fillna("(none)"), margins=True))
print(f"calls this run: {RATE['calls']}, HTTP 429: {RATE['n429']}, network errors: {RATE['network_errors']}, min ratelimit-remaining seen: {RATE['min_remaining']}")


eligible rows 123,965 = 124,858 − 893 (600 sampled ∪ 294 edge rows, 1 in both); sampled 2000 with seed 20261029

,rows_in_data,sampled_2000,row_share,expected_proportional,sample_share
chain,,,,,
base,34908,574,0.2796,559.2,0.2870
ethereum,29351,477,0.2351,470.2,0.2385
bnb,27052,411,0.2167,433.4,0.2055
arbitrum,20398,325,0.1634,326.8,0.1625
polygon,7393,124,0.0592,118.4,0.0620
optimism,5756,89,0.0461,92.2,0.0445


sampled 2,000 by bridge (top 12): {'across': 596, 'relaydepository': 470, 'lifiIntents': 248, 'layerswap': 203, 'mayan': 189, 'polymerStandard': 92, 'gasZipBridge': 60, 'near': 47, 'polymer': 24, 'mayanFastMCTP': 17, 'symbiosis': 10, 'lifiIntentsDex': 9}


lifi_api_transfer_status_sample_2600.csv: read snapshot, 2600 rows, fetched 2026-10-28T19:07:08+00:00 (no network call)


lifi_api_transfer_status_sample_2600.csv: (2600, 71); by design {'uniform_2000': 2000, 'equal_per_chain_600': 600}; HTTP statuses {np.int64(200): 2600}
status/substatus by design:


,substatus,COMPLETED,PARTIAL,REFUNDED,All
sample_design,status,,,,
equal_per_chain_600,DONE,593,1,6,600
uniform_2000,DONE,1964,6,30,2000
All,,2557,7,36,2600


calls this run: 0, HTTP 429: 0, network errors: 0, min ratelimit-remaining seen: None


## 7. Completion rates on the uniform 2,000

`u` is the uniform 2,000, `e600` the equal-per-chain 600 (01 §5), `s2600` both. Rates are `DONE` and `COMPLETED` over sampled events, with the Wilson 95% interval; groups need at least 30 sampled events (`MIN_N`).

### 7.1 Overall and by source chain

In [14]:
u = s2600[s2600["sample_design"] == "uniform_2000"].copy()
e600 = s2600[s2600["sample_design"] == "equal_per_chain_600"].copy()
for frame in (u, e600, s2600):
    frame["done"] = frame["status"] == "DONE"
    frame["completed"] = frame["done"] & (frame["substatus"] == "COMPLETED")
    frame["refunded"] = frame["done"] & (frame["substatus"] == "REFUNDED")
    frame["partial"] = frame["done"] & (frame["substatus"] == "PARTIAL")


def rate_table(frame, by, min_n=1):
    "k completed of n sampled per group with the Wilson 95% interval; groups with fewer than min_n sampled are dropped."
    t = frame.groupby(by).agg(sampled=("completed", "size"), completed=("completed", "sum"), refunded=("refunded", "sum"), partial=("partial", "sum"), not_done=("done", lambda s: int((~s).sum())))
    t = t[t["sampled"] >= min_n].copy()
    t["rate"] = (t["completed"] / t["sampled"]).round(4)
    iv = [wilson(k, n) for k, n in zip(t["completed"], t["sampled"])]
    t["wilson_low"], t["wilson_high"] = [a for a, _ in iv], [b for _, b in iv]
    return t.sort_values("sampled", ascending=False)


n, k = len(u), int(u["completed"].sum())
print(f"uniform 2,000: HTTP statuses {u['http_status'].value_counts().to_dict()}; status DONE {int(u['done'].sum())} of {n}")
print(f"completed (DONE and COMPLETED): {k} of {n} = {k / n:.4f}, Wilson 95% {wilson(k, n)}; refunded {int(u['refunded'].sum())}, partial {int(u['partial'].sum())}, other {n - k - int(u['refunded'].sum()) - int(u['partial'].sum())}")
display(pd.crosstab(u["status"].fillna("(none)"), u["substatus"].fillna("(none)"), margins=True))
print("by source chain (uniform 2,000):")
by_chain = rate_table(u, "chain")
by_chain["rows_in_data"] = tf["chain"].value_counts()
display(by_chain)

uniform 2,000: HTTP statuses {np.int64(200): 2000}; status DONE 2000 of 2000
completed (DONE and COMPLETED): 1964 of 2000 = 0.9820, Wilson 95% (0.9752, 0.987); refunded 30, partial 6, other 0


substatus,COMPLETED,PARTIAL,REFUNDED,All
status,,,,
DONE,1964,6,30,2000
All,1964,6,30,2000


by source chain (uniform 2,000):


,sampled,completed,refunded,partial,not_done,rate,wilson_low,wilson_high,rows_in_data
chain,,,,,,,,,
base,574,566,6,2,0,0.9861,0.9727,0.9929,34908
ethereum,477,461,13,3,0,0.9665,0.9462,0.9792,29351
bnb,411,404,7,0,0,0.9830,0.9653,0.9917,27052
arbitrum,325,320,4,1,0,0.9846,0.9645,0.9934,20398
polygon,124,124,0,0,0,1.0000,0.9700,1.0000,7393
optimism,89,89,0,0,0,1.0000,0.9586,1.0000,5756


### 7.2 By bridge and by integrator (at least 30 sampled events)

In [15]:
MIN_N = 30
print(f"by bridge, bridges with at least {MIN_N} sampled events (uniform 2,000); rows_in_data and usd_provided are the bridge's totals in the data:")
by_bridge = rate_table(u, "event_bridge", MIN_N)
by_bridge["rows_in_data"] = tf["bridge"].value_counts()
by_bridge["usd_provided_in_data"] = tf.groupby("bridge")["amount_usd_provided"].sum().round(0)
display(by_bridge)
print("bridges under the threshold, pooled:", u[~u["event_bridge"].isin(by_bridge.index)].groupby("event_bridge")["completed"].agg(["size", "sum"]).rename(columns={"size": "sampled", "sum": "completed"}).sort_values("sampled", ascending=False).T.to_dict())
print(f"by integrator, integrators with at least {MIN_N} sampled events (uniform 2,000):")
by_integrator = rate_table(u, "event_integrator", MIN_N)
by_integrator["rows_in_data"] = tf["integrator"].value_counts()
display(by_integrator)
print("integrators under the threshold, pooled:", int((~u["event_integrator"].isin(by_integrator.index)).sum()), "events,", int(u[~u["event_integrator"].isin(by_integrator.index)]["completed"].sum()), "completed")

by bridge, bridges with at least 30 sampled events (uniform 2,000); rows_in_data and usd_provided are the bridge's totals in the data:


,sampled,completed,refunded,partial,not_done,rate,wilson_low,wilson_high,rows_in_data,usd_provided_in_data
event_bridge,,,,,,,,,,
across,596,594,0,2,0,0.9966,0.9878,0.9991,34163,24717867.0
relaydepository,470,459,11,0,0,0.9766,0.9586,0.9869,31483,34025726.0
lifiIntents,248,242,5,1,0,0.9758,0.9482,0.9889,16185,40167170.0
layerswap,203,196,7,0,0,0.9655,0.9305,0.9832,12659,7577968.0
mayan,189,185,3,1,0,0.9788,0.9469,0.9917,11422,19193985.0
polymerStandard,92,92,0,0,0,1.0000,0.9599,1.0000,6631,129702294.0
gasZipBridge,60,60,0,0,0,1.0000,0.9398,1.0000,3678,20303.0
near,47,46,1,0,0,0.9787,0.8889,0.9962,2814,11369464.0


bridges under the threshold, pooled: {'polymer': {'sampled': 24, 'completed': 24}, 'mayanFastMCTP': {'sampled': 17, 'completed': 17}, 'symbiosis': {'sampled': 10, 'completed': 9}, 'glacis': {'sampled': 9, 'completed': 9}, 'lifiIntentsDex': {'sampled': 9, 'completed': 6}, 'stargateV2': {'sampled': 8, 'completed': 8}, 'squid': {'sampled': 6, 'completed': 5}, 'mayanMCTP': {'sampled': 5, 'completed': 5}, 'arbitrum': {'sampled': 2, 'completed': 2}, 'celercircle': {'sampled': 1, 'completed': 1}, 'm0': {'sampled': 1, 'completed': 1}, 'omni': {'sampled': 1, 'completed': 1}, 'paxosDex': {'sampled': 1, 'completed': 1}, 'polygon': {'sampled': 1, 'completed': 1}}
by integrator, integrators with at least 30 sampled events (uniform 2,000):


,sampled,completed,refunded,partial,not_done,rate,wilson_low,wilson_high,rows_in_data
event_integrator,,,,,,,,,
jumper.exchange,645,637,8,0,0,0.9876,0.9757,0.9937,40908
_binancewallet,229,227,2,0,0,0.9913,0.9687,0.9976,15109
rabbykmsv2,140,136,4,0,0,0.9714,0.9288,0.9888,7759
metamask-bridge,128,126,1,1,0,0.9844,0.9448,0.9957,7332
phantom,92,90,1,1,0,0.9783,0.9242,0.9940,5233
base-app,86,84,2,0,0,0.9767,0.9191,0.9936,5859
lifi-api,46,44,1,1,0,0.9565,0.8547,0.9880,3465
amahub,42,42,0,0,0,1.0000,0.9162,1.0000,1875
bitget,41,41,0,0,0,1.0000,0.9143,1.0000,2534


integrators under the threshold, pooled: 415 events, 404 completed


### 7.3 Does the response describe the sampled event?

In [16]:
# Does the response describe the sampled event? Identity checks on the uniform 2,000, as 01 §10 did on the 600.
u["integrator_matches"] = u["metadata_integrator"].fillna("") == u["event_integrator"].fillna("")
u["tool_matches"] = u["tool"] == u["event_bridge"]
u["sending_chain_matches"] = u["sending_chainId"] == u["chain"].map(SOURCE_CHAIN_ID)
u["dest_chain_matches"] = u["receiving_chainId"] == u["event_destinationChainId"]
checks = {c: int(u[c].fillna(False).astype(bool).sum()) for c in ["sending_hash_matches", "transaction_id_matches", "integrator_matches", "tool_matches", "sending_chain_matches", "dest_chain_matches"]}
print("matches of", n, ":", checks)
print("transactionId mismatches:")
display(u[~u["transaction_id_matches"].fillna(False).astype(bool)][["chain", "tx_hash", "log_index", "event_transactionId", "api_transactionId", "event_bridge", "tool", "status", "substatus", "batch_n", "batch_calls_n"]])
print("tool differs from the event's bridge string:")
display(u[~u["tool_matches"].fillna(False)][["chain", "event_bridge", "tool", "status", "substatus"]].value_counts().to_frame("rows"))
print("receiving chain differs from destinationChainId (refunds land on the source chain; unlisted ids are reported under another id):")
display(u[~u["dest_chain_matches"].fillna(False)].groupby(["event_destinationChainId", "receiving_chainId", "substatus"], dropna=False).size().to_frame("rows"))
print("responses carrying batchStatus:", int(u["batch_n"].notna().sum()), " with more than one entry:", int((u["batch_n"].fillna(0) > 1).sum()))
u["event_id_in_batch"] = [str(ids or "").lower().find(str(t).lower()) >= 0 if pd.notna(ids) else None for ids, t in zip(u["batch_call_transactionIds"], u["event_transactionId"])]
print("responses with more than one batchStatus entry (single-event transactions in the data whose status response lists several user operations):")
display(u[u["batch_n"].fillna(0) > 1][["chain", "tx_hash", "log_index", "event_bridge", "event_integrator", "event_transactionId", "api_transactionId", "transaction_id_matches", "event_id_in_batch", "batch_n", "batch_calls_n", "batch_call_tools", "batch_call_statuses", "status", "substatus"]])
print("transactionId mismatches whose event id appears in batchStatus:", int(u[~u["transaction_id_matches"].fillna(False).astype(bool)]["event_id_in_batch"].fillna(False).astype(bool).sum()), "of", int((~u["transaction_id_matches"].fillna(False).astype(bool)).sum()))

matches of 2000 : {'sending_hash_matches': 2000, 'transaction_id_matches': 1994, 'integrator_matches': 2000, 'tool_matches': 2000, 'sending_chain_matches': 2000, 'dest_chain_matches': 1964}
transactionId mismatches:


,chain,tx_hash,log_index,event_transactionId,api_transactionId,event_bridge,tool,status,substatus,batch_n,batch_calls_n
381,base,0x1eac51f2fd2454f7fd68ee405ce478a976d6792995c056457312b30858ffcafe,517,0xb6062fa23f2e2b8cc6c0a6ab7a6a8655ec022b26693ac44e9c786069d3640ef9,0xea516a980edc4957fce7b7950331d1e4a6780deddaaca90a3597d4a09a36752c,mayan,mayan,DONE,COMPLETED,2,2
382,base,0x9f1f3428dd9fe0b38d3aea69de36180f1c07c58f5a79ed97be56a230337253f5,337,0x236ac39cf1941e8d7b2a64532634098655af9c65c4a0a02038ea94d042aa1f2e,0x0bc9f7f1a407bdb79658b0dced7facd8bd4f2030893fda9466c40792194df4c6,relaydepository,relaydepository,DONE,COMPLETED,2,2
404,base,0xad17592ca64865420f4f98ed6c38d5e19d1a9d894492d9e47557d9cc6a26cf43,304,0xceb41d9d89b96eb14263c15adae22b33c451555b9564542b01e53195dc008017,0xf87592b8b07815f9ef0e804705302723fa027481abb4bcff54649a94bae42b6a,gasZipBridge,gasZipBridge,DONE,COMPLETED,4,4
571,base,0xfbde97ea9b5deac252b29d1e19110fd3e4217439f0ea90044750dd2f552bb007,103,0x451532f3c17a9f0c6d3a98f1cefe464d9910e74b47680ec8a1ac22f5563797b7,0x4c5270569d357dc32d7178af01e00d5ef3e400405672a0e53c10e8304e89cf74,gasZipBridge,gasZipBridge,DONE,COMPLETED,2,2
823,base,0x12298e0212c222d1fa1ab7d328f19134a0677b0b10db1b7b4bea828f5cde99e9,360,0x02a84eabb0ce3cc5e77a646f9bf1e136405dbcfe07768cedc14c659ec6423830,0x6ea00bba2ffb3b70646cfdc0083c126d29823c71cd0aaf2d67c2b55b01a1aeb4,relaydepository,relaydepository,DONE,COMPLETED,5,5
860,base,0x84abee575dc9822f674e2e2933e300823561ac9ba17779750e4124e87f5ff91f,131,0x5ad24400b82e368485d759353ca768961404af96edb1315a2b849170fc2c52cc,0xd19fbb8dae2b2ff188d20a40a4f3e5e7cd01f755f2f2fac4a5494174485e9489,lifiIntents,lifiIntents,DONE,COMPLETED,2,2


tool differs from the event's bridge string:


,,,,,rows
chain,event_bridge,tool,status,substatus,


receiving chain differs from destinationChainId (refunds land on the source chain; unlisted ids are reported under another id):


rows
event_destinationChainId receiving_chainId substatus      
1                        56                REFUNDED      2
                         8453              REFUNDED      1
                         42161             REFUNDED      1
25                       56                REFUNDED      1
56                       1                 REFUNDED      6
                         8453              REFUNDED      1
100                      1                 REFUNDED      1
                         56                REFUNDED      1
137                      1                 REFUNDED      3
                         42161             REFUNDED      1
4663                     1                 REFUNDED      1
                         8453              REFUNDED      1
8453                     56                REFUNDED      1
                         42161             REFUNDED      1
42161                    1                 REFUNDED      1
1151111081099710         56                REFUNDED      1
                         8453              REFUNDED      2
                         42161             REFUNDED      1
1885080386571452         728126428         COMPLETED     9

responses carrying batchStatus: 260  with more than one entry: 6
responses with more than one batchStatus entry (single-event transactions in the data whose status response lists several user operations):


,chain,tx_hash,log_index,event_bridge,event_integrator,event_transactionId,api_transactionId,transaction_id_matches,event_id_in_batch,batch_n,batch_calls_n,batch_call_tools,batch_call_statuses,status,substatus
381,base,0x1eac51f2fd2454f7fd68ee405ce478a976d6792995c056457312b30858ffcafe,517,mayan,base-app,0xb6062fa23f2e2b8cc6c0a6ab7a6a8655ec022b26693ac44e9c786069d3640ef9,0xea516a980edc4957fce7b7950331d1e4a6780deddaaca90a3597d4a09a36752c,False,True,2,2,fly|mayan,DONE/COMPLETED|DONE/COMPLETED,DONE,COMPLETED
382,base,0x9f1f3428dd9fe0b38d3aea69de36180f1c07c58f5a79ed97be56a230337253f5,337,relaydepository,base-app,0x236ac39cf1941e8d7b2a64532634098655af9c65c4a0a02038ea94d042aa1f2e,0x0bc9f7f1a407bdb79658b0dced7facd8bd4f2030893fda9466c40792194df4c6,False,True,2,2,relaydepository|fly,DONE/COMPLETED|DONE/COMPLETED,DONE,COMPLETED
404,base,0xad17592ca64865420f4f98ed6c38d5e19d1a9d894492d9e47557d9cc6a26cf43,304,gasZipBridge,base-app,0xceb41d9d89b96eb14263c15adae22b33c451555b9564542b01e53195dc008017,0xf87592b8b07815f9ef0e804705302723fa027481abb4bcff54649a94bae42b6a,False,True,4,4,fly|fly|kyberswap|gasZipBridge,DONE/COMPLETED|DONE/COMPLETED|DONE/COMPLETED|DONE/COMPLETED,DONE,COMPLETED
571,base,0xfbde97ea9b5deac252b29d1e19110fd3e4217439f0ea90044750dd2f552bb007,103,gasZipBridge,base-app,0x451532f3c17a9f0c6d3a98f1cefe464d9910e74b47680ec8a1ac22f5563797b7,0x4c5270569d357dc32d7178af01e00d5ef3e400405672a0e53c10e8304e89cf74,False,True,2,2,gasZipBridge|fly,DONE/COMPLETED|DONE/COMPLETED,DONE,COMPLETED
823,base,0x12298e0212c222d1fa1ab7d328f19134a0677b0b10db1b7b4bea828f5cde99e9,360,relaydepository,base-app,0x02a84eabb0ce3cc5e77a646f9bf1e136405dbcfe07768cedc14c659ec6423830,0x6ea00bba2ffb3b70646cfdc0083c126d29823c71cd0aaf2d67c2b55b01a1aeb4,False,True,5,5,kyberswap|fly|relaydepository|kyberswap|kyberswap,DONE/COMPLETED|DONE/COMPLETED|DONE/COMPLETED|DONE/COMPLETED|DONE/C...,DONE,COMPLETED
860,base,0x84abee575dc9822f674e2e2933e300823561ac9ba17779750e4124e87f5ff91f,131,lifiIntents,base-app,0x5ad24400b82e368485d759353ca768961404af96edb1315a2b849170fc2c52cc,0xd19fbb8dae2b2ff188d20a40a4f3e5e7cd01f755f2f2fac4a5494174485e9489,False,True,2,2,lifiIntents|fly,DONE/COMPLETED|DONE/COMPLETED,DONE,COMPLETED


transactionId mismatches whose event id appears in batchStatus: 6 of 6


### 7.4 The uniform 2,000 against the 600 equal-per-chain sample

In [17]:
# The uniform 2,000 against the 600 equal-per-chain sample on the overall completed rate (01 §10), and both pooled.
rows = []
for label, frame in [("uniform_2000", u), ("equal_per_chain_600", e600), ("pooled_2600", s2600)]:
    kk, nn = int(frame["completed"].sum()), len(frame)
    rows.append({"sample": label, "n": nn, "completed": kk, "rate": round(kk / nn, 4), "wilson_low": wilson(kk, nn)[0], "wilson_high": wilson(kk, nn)[1],
                 "refunded": int(frame["refunded"].sum()), "partial": int(frame["partial"].sum())})
cmp = pd.DataFrame(rows).set_index("sample")
share = tf["chain"].value_counts(normalize=True)
r600 = e600.groupby("chain")["completed"].mean()
cmp.loc["equal_per_chain_600", "row_weighted_rate"] = round(float((r600 * share).sum()), 4)
display(cmp)
p1, n1, p2, n2 = cmp.loc["uniform_2000", "rate"], 2000, cmp.loc["equal_per_chain_600", "rate"], 600
pp = (p1 * n1 + p2 * n2) / (n1 + n2)
z = (p1 - p2) / math.sqrt(pp * (1 - pp) * (1 / n1 + 1 / n2))
print(f"difference uniform − equal-per-chain: {p1 - p2:+.4f}; two-proportion z = {z:.2f} (|z| < 1.96 means the two designs do not differ at the 5% level); intervals overlap: {cmp.loc['uniform_2000', 'wilson_low'] <= cmp.loc['equal_per_chain_600', 'wilson_high'] and cmp.loc['equal_per_chain_600', 'wilson_low'] <= cmp.loc['uniform_2000', 'wilson_high']}")
print("per chain, both designs:")
display(pd.concat({"uniform_2000": rate_table(u, "chain")[["sampled", "completed", "rate"]], "equal_per_chain_600": rate_table(e600, "chain")[["sampled", "completed", "rate"]]}, axis=1))
# ---- §8 refunds and partials

,n,completed,rate,wilson_low,wilson_high,refunded,partial,row_weighted_rate
sample,,,,,,,,
uniform_2000,2000,1964,0.9820,0.9752,0.9870,30,6,NaN
equal_per_chain_600,600,593,0.9883,0.9761,0.9943,6,1,0.9836
pooled_2600,2600,2557,0.9835,0.9778,0.9877,36,7,NaN


difference uniform − equal-per-chain: -0.0063; two-proportion z = -1.06 (|z| < 1.96 means the two designs do not differ at the 5% level); intervals overlap: True
per chain, both designs:


uniform_2000                   equal_per_chain_600                
              sampled completed    rate             sampled completed  rate
chain                                                                      
base              574       566  0.9861                 100        98  0.98
ethereum          477       461  0.9665                 100        97  0.97
bnb               411       404  0.9830                 100        99  0.99
arbitrum          325       320  0.9846                 100        99  0.99
polygon           124       124  1.0000                 100       100  1.00
optimism           89        89  1.0000                 100       100  1.00

## 8. Refunds and partials (all 2,600)

Every sampled event whose substatus is not `COMPLETED`: the 2,000 plus the 6 refunds and 1 partial already in the 600 (read from the snapshot, not refetched).

In [18]:
nc = s2600[~s2600["completed"]].copy()
for c in ("sending_timestamp", "receiving_timestamp"):
    nc[c] = pd.to_numeric(nc[c], errors="coerce")
nc["hours_to_receiving"] = ((nc["receiving_timestamp"] - nc["sending_timestamp"]) / 3600).round(2)
nc["receiving_on_source_chain"] = nc["receiving_chainId"] == nc["chain"].map(SOURCE_CHAIN_ID)
nc["recv_over_send"] = (nc["receiving_amountUSD"] / nc["sending_amountUSD"]).round(4)
print(f"not COMPLETED in the 2,600: {len(nc)} ({int(nc['refunded'].sum())} REFUNDED, {int(nc['partial'].sum())} PARTIAL, {int((~nc['done']).sum())} not DONE); by design {nc['sample_design'].value_counts().to_dict()}")
display(nc[["sample_design", "chain", "block_time", "event_bridge", "event_integrator", "event_destinationChainId", "sending_token_symbol", "sending_amountUSD", "amount_usd_provided", "receiving_chainId", "receiving_on_source_chain",
            "receiving_token_symbol", "receiving_amountUSD", "recv_over_send", "hours_to_receiving", "event_hasSourceSwaps", "event_hasDestinationCall", "status", "substatus", "substatusMessage"]].round(2))

not COMPLETED in the 2,600: 43 (36 REFUNDED, 7 PARTIAL, 0 not DONE); by design {'uniform_2000': 36, 'equal_per_chain_600': 7}


,sample_design,chain,block_time,event_bridge,event_integrator,event_destinationChainId,sending_token_symbol,sending_amountUSD,amount_usd_provided,receiving_chainId,receiving_on_source_chain,receiving_token_symbol,receiving_amountUSD,recv_over_send,hours_to_receiving,event_hasSourceSwaps,event_hasDestinationCall,status,substatus,substatusMessage
12,uniform_2000,arbitrum,2026-10-13T11:00:00,mayan,jumper.exchange,1151111081099710,USDC,478260.21,477868.67,42161,True,USDC,478167.60,1.00,0.27,False,False,DONE,REFUNDED,The tokens were refunded to the user.
118,uniform_2000,arbitrum,2026-10-15T12:49:29,mayan,_arbitrum,1,USDC,38110.93,38107.89,1,False,USDC,38069.63,1.00,0.57,False,False,DONE,PARTIAL,"The bridge leg delivered, but the destination swap was refunded — ..."
204,uniform_2000,arbitrum,2026-10-17T10:03:25,lifiIntents,jumper.exchange,137,USDC,99.93,100.02,42161,True,USDC,99.92,1.00,0.21,False,False,DONE,REFUNDED,The tokens were refunded to the user.
223,uniform_2000,arbitrum,2026-10-17T21:56:22,layerswap,jumper.exchange,8453,ETH,0.02,0.02,42161,True,ETH,0.02,0.95,0.00,True,False,DONE,REFUNDED,The tokens were refunded to the user.
314,uniform_2000,arbitrum,2026-10-19T19:21:41,relaydepository,metamask-bridge,1,ETH,641.73,640.60,42161,True,USDC,640.92,1.00,0.00,True,False,DONE,REFUNDED,The tokens were refunded to the user.
506,uniform_2000,base,2026-10-14T19:47:43,relaydepository,lifi-api,1151111081099710,USDC,1.20,1.20,8453,True,USDC,1.19,1.00,0.00,True,False,DONE,REFUNDED,The tokens were refunded to the user.
573,uniform_2000,base,2026-10-15T18:06:11,across,metamask-bridge,137,USDC,52.71,52.50,137,False,USDC.e,52.44,0.99,0.00,True,True,DONE,PARTIAL,Some of the received tokens are not the requested destination tokens.
630,uniform_2000,base,2026-10-16T14:32:43,lifiIntentsDex,baibai-prod,8453,WETH,16.82,16.74,8453,True,WETH,16.78,1.00,0.18,True,False,DONE,REFUNDED,The tokens were refunded to the user.
739,uniform_2000,base,2026-10-18T01:54:19,lifiIntents,base-app,56,USDC,522.80,514.93,8453,True,USDC,517.57,0.99,0.21,True,False,DONE,REFUNDED,The tokens were refunded to the user.
769,uniform_2000,base,2026-10-18T13:05:25,lifiIntents,lifi-api,4663,USDC,600.49,598.63,4663,False,USDG,598.07,1.00,0.21,True,False,DONE,PARTIAL,"The bridge leg delivered, but the destination swap was refunded — ..."


In [19]:
print("refunds and partials against the sampled population (2,600), by bridge, chain, token and size:")
pat = s2600.groupby("event_bridge").agg(sampled=("completed", "size"), refunded=("refunded", "sum"), partial=("partial", "sum"))
pat = pat[(pat["refunded"] + pat["partial"]) > 0].assign(refund_rate=lambda d: (d["refunded"] / d["sampled"]).round(4))
display(pat.sort_values("sampled", ascending=False))
display(s2600.groupby("chain").agg(sampled=("completed", "size"), refunded=("refunded", "sum"), partial=("partial", "sum")).assign(refund_rate=lambda d: (d["refunded"] / d["sampled"]).round(4)))
print("refunded transfers by sending token symbol:", nc[nc["refunded"]]["sending_token_symbol"].value_counts().to_dict())
print("refunded transfers by destination id:", nc[nc["refunded"]]["event_destinationChainId"].value_counts().to_dict())
sizes = pd.DataFrame({"refunded": nc[nc["refunded"]]["sending_amountUSD"].describe(percentiles=[0.25, 0.5, 0.75]), "completed (2,600)": s2600[s2600["completed"]]["sending_amountUSD"].describe(percentiles=[0.25, 0.5, 0.75])}).round(2)
display(sizes.T)
flags = pd.DataFrame({"refunded": nc[nc["refunded"]][["event_hasSourceSwaps", "event_hasDestinationCall"]].mean(), "completed": s2600[s2600["completed"]][["event_hasSourceSwaps", "event_hasDestinationCall"]].mean()}).round(4)
display(flags.T)
print(f"USD refunded over USD sent (API USD on both legs), refunds with both amounts: median {nc[nc['refunded']]['recv_over_send'].median():.4f}, min {nc[nc['refunded']]['recv_over_send'].min():.4f}, max {nc[nc['refunded']]['recv_over_send'].max():.4f}; hours from sending to the refund: median {nc[nc['refunded']]['hours_to_receiving'].median():.2f}, max {nc[nc['refunded']]['hours_to_receiving'].max():.2f}")
# ---- §9 received against sent, gas

refunds and partials against the sampled population (2,600), by bridge, chain, token and size:


,sampled,refunded,partial,refund_rate
event_bridge,,,,
across,775,0,2,0.0000
relaydepository,598,14,0,0.0234
lifiIntents,326,6,1,0.0184
layerswap,270,8,0,0.0296
mayan,243,4,1,0.0165
near,62,1,0,0.0161
symbiosis,14,0,2,0.0000
lifiIntentsDex,9,3,0,0.3333
squid,7,0,1,0.0000


,sampled,refunded,partial,refund_rate
chain,,,,
arbitrum,425,4,2,0.0094
base,674,8,2,0.0119
bnb,511,8,0,0.0157
ethereum,577,16,3,0.0277
optimism,189,0,0,0.0000
polygon,224,0,0,0.0000


refunded transfers by sending token symbol: {'ETH': 13, 'USDC': 10, 'USDT': 5, 'BNB': 3, 'WETH': 1, 'KEYCAT': 1, 'RDWon': 1, 'ODAI': 1, 'WBTC': 1}
refunded transfers by destination id: {np.int64(56): 10, np.int64(137): 6, np.int64(1151111081099710): 5, np.int64(1): 5, np.int64(8453): 3, np.int64(4663): 2, np.int64(100): 2, np.int64(25): 1, np.int64(42161): 1, np.int64(5042): 1}


,count,mean,std,min,25%,50%,75%,max
refunded,36.0,16012.26,80474.92,0.02,4.70,50.40,471.49,478260.21
"completed (2,600)",2557.0,2304.77,27170.79,0.00,5.61,46.73,302.38,1075240.61


,event_hasSourceSwaps,event_hasDestinationCall
refunded,0.7778,0.000
completed,0.6785,0.034


USD refunded over USD sent (API USD on both legs), refunds with both amounts: median 0.9964, min 0.9189, max 1.0007; hours from sending to the refund: median 0.07, max 0.99


## 9. Receiving against sending USD, and gas (uniform 2,000)

### 9.1 Receiving over sending USD on completed transfers (the API's own USD on each leg)

In [20]:
done_ = u[u["completed"]].copy()
done_["recv_over_send"] = done_["receiving_amountUSD"] / done_["sending_amountUSD"]
done_["send_over_provided"] = done_["sending_amountUSD"] / done_["amount_usd_provided"]
print("completed transfers in the uniform 2,000 with both USD amounts:", int(done_["recv_over_send"].notna().sum()), "of", len(done_))
display(done_["recv_over_send"].describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("receiving_usd / sending_usd").T.round(4))
loss = 1 - done_["recv_over_send"]
print(f"median USD lost between sending and receiving: {loss.median():.4%}; share losing more than 1%: {(loss > 0.01).mean():.4f}; more than 5%: {(loss > 0.05).mean():.4f}; more than 10%: {(loss > 0.10).mean():.4f}; receiving more than sent: {(loss < 0).mean():.4f}")
print(f"sum sending USD {done_['sending_amountUSD'].sum():,.2f}  sum receiving USD {done_['receiving_amountUSD'].sum():,.2f}  ratio {done_['receiving_amountUSD'].sum() / done_['sending_amountUSD'].sum():.4f}")
outl = done_[(done_["recv_over_send"] < 0.5) | (done_["recv_over_send"] > 2)]
print(f"ratios below 0.5 or above 2 (the API's USD on a thin receiving token, not a loss in the transfer): {len(outl)} rows")
display(outl[["chain", "event_bridge", "event_destinationChainId", "sending_token_symbol", "sending_amountUSD", "receiving_token_symbol", "receiving_amount", "receiving_token_decimals", "receiving_amountUSD", "recv_over_send"]].round(3))
# Why each of those rows is extreme: the top-level record belongs to another call of a bundle; a tiny transfer whose fees exceed the amount; a thin receiving token as the API prices it; or a sending token the API prices far from the receiving leg.
MAJORS = {"ETH", "WETH", "BNB", "WBNB", "xDAI", "USDC", "USDC.e", "USDT", "DAI", "SOL", "POL", "BTC", "WBTC", "cbBTC"}
recs_u = {r["tx_hash"]: r for r in jsonl_read(S2600_JSONL) if r.get("sample_design") == "uniform_2000"}


def own_call(row):
    "The batchStatus call whose transactionId is the event's: (sending USD, receiving USD), or (None, None)."
    for e in (recs_u[row["tx_hash"]]["response"].get("batchStatus") or []):
        for c in e.get("calls") or []:
            if str(c.get("transactionId") or "").lower() == str(row["event_transactionId"]).lower():
                return float((c.get("sending") or {}).get("amountUSD") or "nan"), float((c.get("receiving") or {}).get("amountUSD") or "nan")
    return None, None


def classify(row):
    s_major, r_major = row["sending_token_symbol"] in MAJORS, row["receiving_token_symbol"] in MAJORS
    if pd.notna(row["transaction_id_matches"]) and not bool(row["transaction_id_matches"]):
        return "bundle: top-level record is another call"
    if s_major and r_major and row["sending_amountUSD"] < 0.05:
        return "tiny transfer: quoted fees are most of the amount"
    if not r_major:
        return "thin receiving token as the API prices it"
    if not s_major:
        return "sending token priced off by the API"
    return "unclassified"


outl = outl.copy()
outl["class"] = outl.apply(classify, axis=1)
outl["own_call_sending_usd"], outl["own_call_receiving_usd"] = zip(*outl.apply(own_call, axis=1))
outl["own_call_ratio"] = (outl["own_call_receiving_usd"] / outl["own_call_sending_usd"]).round(4)
outl["quoted_fee_share"] = (outl["quote_fee_usd"] / outl["sending_amountUSD"]).round(3)   # the quote's fee lines over the sending USD
display(outl.sort_values("recv_over_send")[["chain", "event_bridge", "event_integrator", "sending_token_symbol", "sending_amountUSD", "receiving_token_symbol", "receiving_amountUSD", "recv_over_send", "quote_toAmountUSD", "quote_fee_usd", "quoted_fee_share",
                                             "transaction_id_matches", "batch_calls_n", "own_call_sending_usd", "own_call_receiving_usd", "own_call_ratio", "class"]].round(4))
print("rows per class:", outl["class"].value_counts().to_dict())
tiny = outl[outl["class"].str.startswith("tiny")]
print("tiny transfers: quoted fee USD over sending USD per row:", [f"{f:.4f} of {a:.4f} = {sh:.0%}" for f, a, sh in zip(tiny["quote_fee_usd"], tiny["sending_amountUSD"], tiny["quoted_fee_share"])],
      f"; range {tiny['quoted_fee_share'].min():.0%} to {tiny['quoted_fee_share'].max():.0%}")
print("thin receiving tokens:", sorted(outl[outl["class"].str.startswith("thin")]["receiving_token_symbol"].unique().tolist()), "| by bridge and destination:", outl[outl["class"].str.startswith("thin")].groupby(["event_bridge", "event_destinationChainId"]).size().to_dict())
trimmed = done_[(done_["recv_over_send"] >= 0.5) & (done_["recv_over_send"] <= 2)]
print(f"without them: n {len(trimmed)}, median {trimmed['recv_over_send'].median():.4f}, mean {trimmed['recv_over_send'].mean():.4f}, p5 {trimmed['recv_over_send'].quantile(0.05):.4f}, p95 {trimmed['recv_over_send'].quantile(0.95):.4f}, sums ratio {trimmed['receiving_amountUSD'].sum() / trimmed['sending_amountUSD'].sum():.4f}")
print("by bridge (trimmed, bridges with at least 10 completed):")
display(trimmed.groupby("event_bridge")["recv_over_send"].agg(n="size", median="median", mean="mean", p10=lambda s: s.quantile(0.1), p90=lambda s: s.quantile(0.9), min="min").query("n >= 10").round(4).sort_values("n", ascending=False))
print("by sending USD band (trimmed):")
bands = pd.cut(trimmed["sending_amountUSD"], [0, 10, 100, 1_000, 10_000, 100_000, 1e9], labels=["<$10", "$10–100", "$100–1k", "$1k–10k", "$10k–100k", ">$100k"])
display(trimmed.groupby(bands, observed=True)["recv_over_send"].agg(n="size", median="median", p10=lambda s: s.quantile(0.1)).round(4))

completed transfers in the uniform 2,000 with both USD amounts: 1964 of 1964


,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
receiving_usd / sending_usd,1964.0,42.8678,1107.792,0.0258,0.7825,0.9321,0.9628,0.9876,0.9965,0.9994,1.0014,1.0034,1.0627,37608.3325


median USD lost between sending and receiving: 0.3509%; share losing more than 1%: 0.2872; more than 5%: 0.0698; more than 10%: 0.0270; receiving more than sent: 0.1838
sum sending USD 3,172,707.02  sum receiving USD 3,311,108.44  ratio 1.0436
ratios below 0.5 or above 2 (the API's USD on a thin receiving token, not a loss in the transfer): 20 rows


,chain,event_bridge,event_destinationChainId,sending_token_symbol,sending_amountUSD,receiving_token_symbol,receiving_amount,receiving_token_decimals,receiving_amountUSD,recv_over_send
292,arbitrum,relaydepository,1,ETH,0.036,ETH,4514569262210,18,0.011,0.306
346,base,gasZipBridge,1,WMC,45.167,ETH,463805808409189,18,1.167,0.026
381,base,mayan,1151111081099710,ETH,0.000,SOL,115512650,9,12.279,30698.250
404,base,gasZipBridge,1,ETH,0.000,ETH,52226092299344,18,0.131,261.800
435,base,across,4663,USDC,5.002,WBROO,3378526811937975529166262,18,1.825,0.365
464,base,across,4663,ETH,5.335,FalconMargin,1717951356892883394056031,18,29.587,5.546
514,base,relaydepository,1151111081099710,USDC,2.092,TSF,402154343949,6,256.073,122.388
803,base,mayan,137,ETH,20.198,XLS,3886957312429839089664,18,40760.842,2018.013
823,base,relaydepository,1151111081099710,ETH,0.000,1,3192458226,6,1.317,3292.500
853,base,mayan,137,ETH,1.215,XLS,239693491151806922752,18,2513.562,2069.457


,chain,event_bridge,event_integrator,sending_token_symbol,sending_amountUSD,receiving_token_symbol,receiving_amountUSD,recv_over_send,quote_toAmountUSD,quote_fee_usd,quoted_fee_share,transaction_id_matches,batch_calls_n,own_call_sending_usd,own_call_receiving_usd,own_call_ratio,class
346,base,gasZipBridge,rh-xchain-fees-enabled,WMC,45.1669,ETH,1.1667,0.0258,1.1631,0.3839,0.008,True,1,1.0986,1.0919,0.9939,sending token priced off by the API
968,bnb,relaydepository,jumper.exchange,BNB,0.0222,xDAI,0.0020,0.0901,0.0020,0.0204,0.919,True,<NA>,NaN,NaN,NaN,tiny transfer: quoted fees are most of the amount
1304,bnb,relaydepository,jumper.exchange,BNB,0.0291,xDAI,0.0088,0.3024,0.0088,0.0204,0.701,True,<NA>,NaN,NaN,NaN,tiny transfer: quoted fees are most of the amount
292,arbitrum,relaydepository,gas-sweeper,ETH,0.0360,ETH,0.0110,0.3056,0.0110,0.0251,0.697,True,<NA>,NaN,NaN,NaN,tiny transfer: quoted fees are most of the amount
435,base,across,base-app,USDC,5.0023,WBROO,1.8247,0.3648,1.8247,0.0935,0.019,True,1,4.9870,0.0206,0.0041,thin receiving token as the API prices it
1651,ethereum,mayanFastMCTP,Web3-Wallet,ORAI,298.4484,SQR,1058.9509,3.5482,1092.5181,2.0891,0.007,True,<NA>,NaN,NaN,NaN,thin receiving token as the API prices it
464,base,across,phantom,ETH,5.3346,FalconMargin,29.5868,5.5462,29.6111,0.1165,0.022,True,1,5.0198,0.0442,0.0088,thin receiving token as the API prices it
1780,ethereum,across,phantom,ETH,10.0031,W4WORK,204.3475,20.4284,204.4183,0.0968,0.010,True,<NA>,NaN,NaN,NaN,thin receiving token as the API prices it
1753,ethereum,across,phantom,ETH,2.4002,WMGDN,58.3679,24.3179,58.3700,0.0317,0.013,True,<NA>,NaN,NaN,NaN,thin receiving token as the API prices it
1754,ethereum,across,phantom,ETH,2.9076,WMGDN,82.2713,28.2953,82.8816,0.0828,0.028,True,<NA>,NaN,NaN,NaN,thin receiving token as the API prices it


rows per class: {'thin receiving token as the API prices it': 12, 'tiny transfer: quoted fees are most of the amount': 3, 'bundle: top-level record is another call': 3, 'sending token priced off by the API': 2}
tiny transfers: quoted fee USD over sending USD per row: ['0.0251 of 0.0360 = 70%', '0.0204 of 0.0222 = 92%', '0.0204 of 0.0291 = 70%'] ; range 70% to 92%
thin receiving tokens: ['FalconMargin', 'GMOTH', 'HGLHG', 'SFFINU', 'SQR', 'TSF', 'W4WORK', 'WBROO', 'WMGDN', 'XLS'] | by bridge and destination: {('across', 4663): 8, ('mayan', 137): 2, ('mayanFastMCTP', 1151111081099710): 1, ('relaydepository', 1151111081099710): 1}
without them: n 1944, median 0.9964, mean 0.9862, p5 0.9342, p95 1.0030, sums ratio 0.9984
by bridge (trimmed, bridges with at least 10 completed):


,n,median,mean,p10,p90,min
event_bridge,,,,,,
across,585,0.9975,0.9932,0.9838,1.0011,0.7727
relaydepository,454,0.9875,0.9687,0.9161,1.0003,0.5266
lifiIntents,242,0.9986,0.9982,0.9936,1.0026,0.9309
layerswap,196,0.9968,0.9883,0.9708,0.9999,0.7431
mayan,182,0.9866,0.9767,0.9465,1.0001,0.5807
polymerStandard,92,0.9995,0.9988,0.9944,1.0028,0.9823
gasZipBridge,58,0.9964,0.9938,0.9840,0.9999,0.8929
near,46,0.9969,0.9901,0.9796,1.0011,0.9218
polymer,24,0.9980,0.9962,0.9899,1.0024,0.9728


by sending USD band (trimmed):


,n,median,p10
sending_amountUSD,,,
<$10,619,0.9894,0.9250
$10–100,531,0.9968,0.9728
$100–1k,518,0.9982,0.9830
$1k–10k,229,0.9986,0.9875
$10k–100k,41,0.9994,0.9923
>$100k,6,1.0001,0.9987


### 9.2 The API's sending USD against the provided-feed column; gas USD on both legs

In [21]:
print("the API's sending USD against the provided-feed column, by the source-swap flag (with a swap the API's sending amount is the pre-swap input, the event's the bridged token):")
display(done_.groupby("event_hasSourceSwaps")["send_over_provided"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(4))
ns = done_[~done_["event_hasSourceSwaps"]]["send_over_provided"].dropna()
print(f"no-swap rows: {len(ns)}; beyond 0.5%: {int(((ns - 1).abs() > 0.005).sum())}, beyond 1%: {int(((ns - 1).abs() > 0.01).sum())}, beyond 5%: {int(((ns - 1).abs() > 0.05).sum())}")
print("gas USD on the sending leg (uniform 2,000, all statuses):")
display(u["sending_gasAmountUSD"].describe(percentiles=[0.5, 0.9, 0.99]).to_frame().T.round(4))
display(u.groupby("chain")["sending_gasAmountUSD"].agg(n="count", median="median", mean="mean", p90=lambda s: s.quantile(0.9), total="sum").round(4).sort_values("median", ascending=False))
print("gas USD on the receiving leg (where the response carries it):")
display(u["receiving_gasAmountUSD"].describe(percentiles=[0.5, 0.9, 0.99]).to_frame().T.round(4))
print(f"sending gas as a share of sending USD: median {(u['sending_gasAmountUSD'] / u['sending_amountUSD']).median():.4%}, p90 {(u['sending_gasAmountUSD'] / u['sending_amountUSD']).quantile(0.9):.4%}")
# ---- §10 fees

the API's sending USD against the provided-feed column, by the source-swap flag (with a swap the API's sending amount is the pre-swap input, the event's the bridged token):


,count,mean,std,min,5%,25%,50%,75%,95%,max
event_hasSourceSwaps,,,,,,,,,,
False,624.0,0.9999,0.0024,0.9895,0.9962,0.9989,0.9999,1.0008,1.0035,1.0258
True,1336.0,1.0315,1.0339,0.0000,0.9974,1.0005,1.0027,1.0075,1.0251,38.7369


no-swap rows: 624; beyond 0.5%: 12, beyond 1%: 5, beyond 5%: 0
gas USD on the sending leg (uniform 2,000, all statuses):


,count,mean,std,min,50%,90%,99%,max
sending_gasAmountUSD,2000.0,0.1095,0.3974,0.0001,0.0113,0.2168,2.0857,8.5648


,n,median,mean,p90,total
chain,,,,,
ethereum,477,0.1429,0.4063,1.0228,193.8239
bnb,411,0.0133,0.0215,0.0423,8.8232
arbitrum,325,0.0107,0.0315,0.0359,10.2516
polygon,124,0.0105,0.0168,0.0374,2.0861
base,574,0.0038,0.0069,0.0141,3.9747
optimism,89,0.0001,0.0003,0.0007,0.0293


gas USD on the receiving leg (where the response carries it):


,count,mean,std,min,50%,90%,99%,max
receiving_gasAmountUSD,2000.0,0.0251,0.1013,0.0,0.0057,0.0406,0.3761,2.2361


sending gas as a share of sending USD: median 0.0367%, p90 1.4607%


## 10. Fee lines from the quote (uniform 2,000)

`quote.estimate.feeCosts` is what the quote offered at request time, not what was charged. A line is "LIFI-named" when its `name` starts with `LIFI` (`LIFI Fixed Fee`, `LIFI Shared Fee`); each such line carries a `feeSplit` with `lifiFee` and `integratorFee` in token units, converted here to USD at the line's own `amountUSD / amount` rate. Shares are of the API's sending USD.

### 10.1 LIFI-named lines as a share of sending USD, overall

In [22]:
# Fee lines from the quote (estimate.feeCosts): the lines whose name starts with "LIFI" summed per transfer as a share of the API's sending USD.
fq = u.copy()
fq["has_quote"] = fq["quote_tool"].notna()
fq["has_fee_lines"] = fq["quote_fee_usd"].notna()
fq["has_lifi_line"] = fq["quote_fee_lifi_usd"].fillna(0) > 0
fq["lifi_share"] = fq["quote_fee_lifi_usd"] / fq["sending_amountUSD"]
fq["other_share"] = fq["quote_fee_other_usd"] / fq["sending_amountUSD"]
fq["all_share"] = fq["quote_fee_usd"] / fq["sending_amountUSD"]
print(f"uniform 2,000: with a quote {int(fq['has_quote'].sum())}, with fee lines {int(fq['has_fee_lines'].sum())}, with a LIFI-named line above zero {int(fq['has_lifi_line'].sum())}")
tot_send = fq["sending_amountUSD"].sum()
print(f"sum sending USD {tot_send:,.2f}; all fee lines {fq['quote_fee_usd'].sum():,.2f} ({1e4 * fq['quote_fee_usd'].sum() / tot_send:.1f} bps); LIFI-named lines {fq['quote_fee_lifi_usd'].sum():,.2f} ({1e4 * fq['quote_fee_lifi_usd'].sum() / tot_send:.1f} bps); other lines {fq['quote_fee_other_usd'].sum():,.2f} ({1e4 * fq['quote_fee_other_usd'].sum() / tot_send:.1f} bps)")
print(f"of the LIFI-named USD, feeSplit.lifiFee {fq['fee_split_lifi_usd'].sum():,.2f} and feeSplit.integratorFee {fq['fee_split_integrator_usd'].sum():,.2f}")
print("per-transfer shares of sending USD (rows with fee lines):")
display(fq[fq["has_fee_lines"]][["lifi_share", "other_share", "all_share"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T.round(5))
print("LIFI-named fee percentages (the quote's `percentage` field, one per line) by count of transfers, top 20:")
display(fq["quote_fee_lifi_percentages"].fillna("").replace("", "(no LIFI line)").value_counts().head(20).to_frame("transfers"))
pct_values = sorted({p for v in fq["quote_fee_lifi_percentages"].dropna() for p in str(v).split("|") if p}, key=float)
print(f"distinct percentage values on LIFI-named lines: {len(pct_values)}; transfers with a LIFI line {int(fq['quote_fee_lifi_percentages'].fillna('').ne('').sum())}, without {int(fq['quote_fee_lifi_percentages'].fillna('').eq('').sum())}; values: {pct_values}")


def fee_table(frame, by, min_n=MIN_N):
    t = frame.groupby(by).agg(sampled=("sending_amountUSD", "size"), with_lifi_line=("has_lifi_line", "sum"), sending_usd=("sending_amountUSD", "sum"),
                              lifi_fee_usd=("quote_fee_lifi_usd", "sum"), split_lifi_usd=("fee_split_lifi_usd", "sum"), split_integrator_usd=("fee_split_integrator_usd", "sum"),
                              other_fee_usd=("quote_fee_other_usd", "sum"), lifi_share_median=("lifi_share", "median"),
                              lifi_percentages=("quote_fee_lifi_percentages", lambda s: "|".join(sorted(set(p for v in s.dropna() for p in str(v).split("|") if p)))))
    t = t[t["sampled"] >= min_n].copy()
    t["lifi_bps"] = (1e4 * t["lifi_fee_usd"] / t["sending_usd"]).round(1)
    t["split_lifi_bps"] = (1e4 * t["split_lifi_usd"] / t["sending_usd"]).round(1)
    t["other_bps"] = (1e4 * t["other_fee_usd"] / t["sending_usd"]).round(1)
    t["lifi_share_median"] = t["lifi_share_median"].round(5)
    t["n_lifi_percentages"] = t["lifi_percentages"].str.count(r"\|") + (t["lifi_percentages"] != "").astype(int)
    for c in ["sending_usd", "lifi_fee_usd", "split_lifi_usd", "split_integrator_usd", "other_fee_usd"]:
        t[c] = t[c].round(2)
    return t.sort_values("sending_usd", ascending=False)

uniform 2,000: with a quote 1991, with fee lines 1721, with a LIFI-named line above zero 1213
sum sending USD 3,790,994.81; all fee lines 3,188.58 (8.4 bps); LIFI-named lines 2,523.11 (6.7 bps); other lines 665.47 (1.8 bps)
of the LIFI-named USD, feeSplit.lifiFee 558.20 and feeSplit.integratorFee 1,964.92
per-transfer shares of sending USD (rows with fee lines):


,count,mean,std,min,10%,25%,50%,75%,90%,max
lifi_share,1721.0,0.00288,0.00435,0.00000,0.0000,0.00000,0.00050,0.00300,0.00900,0.02802
other_share,1721.0,0.09073,2.78528,0.00000,0.0000,0.00011,0.00057,0.00339,0.01904,113.00000
all_share,1721.0,0.09361,2.78525,0.00001,0.0002,0.00069,0.00307,0.00947,0.02499,113.00000


LIFI-named fee percentages (the quote's `percentage` field, one per line) by count of transfers, top 20:


,transfers
quote_fee_lifi_percentages,
(no LIFI line),787
0.0025,327
0.0002,259
0.0005,171
0.0085,110
0.0100,81
0.0030,41
0.0090,31
0.0075,26


distinct percentage values on LIFI-named lines: 36; transfers with a LIFI line 1213, without 787; values: ['0.0002', '0.0004', '0.0005', '0.0007', '0.0010', '0.0015', '0.0020', '0.0025', '0.0030', '0.0033', '0.0035', '0.0040', '0.0045', '0.0050', '0.0052', '0.0053', '0.0055', '0.0060', '0.0063', '0.0065', '0.0067', '0.0070', '0.0075', '0.0083', '0.0084', '0.0085', '0.0090', '0.0095', '0.0100', '0.0125', '0.0130', '0.0145', '0.0160', '0.0164', '0.0240', '0.0280']


### 10.2 By integrator and by bridge

In [23]:
print(f"by integrator (at least {MIN_N} sampled events): LIFI-named fee USD over sending USD in bps, the feeSplit's LI.FI part, the other lines, and the distinct LIFI percentages quoted:")
display(fee_table(fq, "event_integrator"))
print(f"by bridge (at least {MIN_N} sampled events):")
display(fee_table(fq, "event_bridge"))

by integrator (at least 30 sampled events): LIFI-named fee USD over sending USD in bps, the feeSplit's LI.FI part, the other lines, and the distinct LIFI percentages quoted:


,sampled,with_lifi_line,sending_usd,lifi_fee_usd,split_lifi_usd,split_integrator_usd,other_fee_usd,lifi_share_median,lifi_percentages,lifi_bps,split_lifi_bps,other_bps,n_lifi_percentages
event_integrator,,,,,,,,,,,,,
jumper.exchange,645,315,1963039.84,76.84,76.84,0.00,266.20,0.00020,0.0002|0.0005,0.4,0.4,1.4,2
_arbitrum,31,0,387318.17,0.00,0.00,0.00,3.78,0.00000,,0.0,0.0,0.1,0
_binancewallet,229,0,292112.82,0.00,0.00,0.00,105.40,0.00000,,0.0,0.0,3.6,0
Cow,32,0,255727.80,0.00,0.00,0.00,127.31,0.00000,,0.0,0.0,5.0,0
phantom,92,92,74686.69,634.82,31.74,603.08,5.06,0.00850,0.0083|0.0084|0.0085,85.0,4.2,0.7,3
rabbykmsv2,140,140,60210.79,150.52,0.00,150.52,5.85,0.00250,0.0025,25.0,0.0,1.0,1
metamask-bridge,128,107,50764.66,19.47,19.47,0.00,25.52,0.00050,0.0004|0.0005,3.8,3.8,5.0,2
lifi-api,46,46,21267.80,53.17,53.17,0.00,5.17,0.00250,0.0025,25.0,25.0,2.4,1
bitget,41,41,16650.29,49.95,0.00,49.95,6.33,0.00300,0.0030,30.0,0.0,3.8,1


by bridge (at least 30 sampled events):


,sampled,with_lifi_line,sending_usd,lifi_fee_usd,split_lifi_usd,split_integrator_usd,other_fee_usd,lifi_share_median,lifi_percentages,lifi_bps,split_lifi_bps,other_bps,n_lifi_percentages
event_bridge,,,,,,,,,,,,,
mayan,189,149,1058878.60,695.57,88.68,606.89,1.41,0.00849,0.0002|0.0005|0.0015|0.0020|0.0025|0.0030|0.0040|0.0050|0.0052|0.0...,6.6,0.8,0.0,23
polymerStandard,92,23,682762.37,33.25,14.54,18.70,0.36,0.00250,0.0010|0.0025|0.0050|0.0075|0.0095|0.0100,0.5,0.2,0.0,6
lifiIntents,248,107,625007.68,454.94,84.29,370.65,0.32,0.00250,0.0002|0.0005|0.0020|0.0025|0.0035|0.0050|0.0055|0.0065|0.0075|0.0...,7.3,1.3,0.0,14
relaydepository,470,308,396034.73,294.33,114.83,179.51,400.81,0.00047,0.0002|0.0005|0.0007|0.0010|0.0020|0.0025|0.0030|0.0035|0.0040|0.0...,7.4,2.9,10.1,24
across,596,310,340010.82,345.74,105.20,240.54,49.26,0.00020,0.0002|0.0004|0.0005|0.0007|0.0010|0.0020|0.0025|0.0030|0.0033|0.0...,10.2,3.1,1.4,26
near,47,32,191046.71,105.87,19.93,85.93,115.96,0.00020,0.0002|0.0005|0.0025|0.0040|0.0070|0.0075|0.0085|0.0090|0.0100|0.0...,5.5,1.0,6.1,12
layerswap,203,185,96390.97,177.53,56.38,121.14,45.74,0.00248,0.0002|0.0005|0.0010|0.0025|0.0035|0.0050|0.0052|0.0053|0.0055|0.0...,18.4,5.8,4.7,22
gasZipBridge,60,43,396.02,1.68,0.43,1.25,0.02,0.00254,0.0002|0.0025|0.0035|0.0055|0.0067|0.0075|0.0085|0.0100|0.0280,42.5,10.9,0.4,9


### 10.3 Every fee line: names, the feeSplit of the LIFI-named lines, the non-LI.FI lines by bridge

In [24]:
# Every fee line of the uniform 2,000, from the raw JSONL: name, count, USD, included flag; the non-LI.FI lines summarised by name and by bridge.
recs2600 = jsonl_read(S2600_JSONL)
lines = []
for r in recs2600:
    if r.get("sample_design") != "uniform_2000":
        continue
    body = r["response"] if isinstance(r["response"], dict) else {}
    ev_bridge = u.loc[u["tx_hash"] == r["tx_hash"], "event_bridge"]
    for f in (((body.get("quote") or {}).get("estimate") or {}).get("feeCosts") or []):
        fs = f.get("feeSplit") or {}
        lines.append({"tx_hash": r["tx_hash"], "bridge": ev_bridge.iloc[0] if len(ev_bridge) else None, "name": f.get("name"), "percentage": f.get("percentage"), "included": f.get("included"),
                      "amountUSD": float(f.get("amountUSD") or 0), "token": (f.get("token") or {}).get("symbol"), "is_lifi": str(f.get("name", "")).startswith("LIFI"),
                      "has_feeSplit": bool(fs), "lifiFee_gt0": int(fs.get("lifiFee") or 0) > 0 if fs else None, "integratorFee_gt0": int(fs.get("integratorFee") or 0) > 0 if fs else None})
fl = pd.DataFrame(lines)
print(f"fee lines: {len(fl)} over {fl['tx_hash'].nunique()} transfers; LIFI-named {int(fl['is_lifi'].sum())}, other {int((~fl['is_lifi']).sum())}; lines with feeSplit {int(fl['has_feeSplit'].sum())} (all LIFI-named: {bool((fl['has_feeSplit'] == fl['is_lifi']).all())})")
by_name = fl.groupby("name").agg(lines=("tx_hash", "size"), transfers=("tx_hash", "nunique"), usd=("amountUSD", "sum"), included_true=("included", lambda s: int(s.fillna(False).astype(bool).sum())),
                                 bridges=("bridge", lambda s: "|".join(sorted(set(map(str, s))))[:80])).sort_values("usd", ascending=False).round(2)
display(by_name)
print("LIFI-named lines: feeSplit shares (lines with lifiFee > 0 / integratorFee > 0):")
display(fl[fl["is_lifi"]].groupby("name").agg(lines=("tx_hash", "size"), lifiFee_gt0=("lifiFee_gt0", "sum"), integratorFee_gt0=("integratorFee_gt0", "sum"), usd=("amountUSD", "sum")).round(2))
print("non-LI.FI lines by bridge: USD and bps of the bridge's sending USD (uniform 2,000):")
nl = fl[~fl["is_lifi"]].groupby("bridge").agg(lines=("tx_hash", "size"), transfers=("tx_hash", "nunique"), usd=("amountUSD", "sum"), names=("name", lambda s: "|".join(sorted(set(s)))[:90]))
nl["sending_usd"] = u.groupby("event_bridge")["sending_amountUSD"].sum()
nl["bps"] = (1e4 * nl["usd"] / nl["sending_usd"]).round(1)
display(nl.sort_values("usd", ascending=False).round(2))

fee lines: 3931 over 1721 transfers; LIFI-named 1213, other 2718; lines with feeSplit 1213 (all LIFI-named: True)


,lines,transfers,usd,included_true,bridges
name,,,,,
LIFI Fixed Fee,985,985,1590.54,985,across|arbitrum|gasZipBridge|glacis|layerswap|lifiIntents|lifiInte...
LIFI Shared Fee,228,228,932.58,228,across|gasZipBridge|layerswap|lifiIntents|mayan|mayanFastMCTP|maya...
Relayer Service fee,468,468,376.96,468,relaydepository
NearIntents App Fee,47,47,77.78,47,near
LayerSwap fee,203,203,45.74,203,layerswap
Relayer fee,595,595,29.90,595,across
Relayer Gas fee,455,455,23.84,455,relaydepository
Airlift Fee,7,7,21.64,7,glacis
NearIntents Protocol Fee,47,47,19.09,47,near


LIFI-named lines: feeSplit shares (lines with lifiFee > 0 / integratorFee > 0):


,lines,lifiFee_gt0,integratorFee_gt0,usd
name,,,,
LIFI Fixed Fee,985,823,366,1590.54
LIFI Shared Fee,228,187,228,932.58


non-LI.FI lines by bridge: USD and bps of the bridge's sending USD (uniform 2,000):


,lines,transfers,usd,names,sending_usd,bps
bridge,,,,,,
relaydepository,925,468,400.81,Relayer Gas fee|Relayer Service fee|Sushi Protocol Fee,396034.73,10.1
near,141,47,115.96,1ClickSwap API Fee|NearIntents App Fee|NearIntents Protocol Fee,191046.71,6.1
across,1314,595,49.26,Execution Gas|LP fee|Network Gas|Relayer fee|Relayer gas fee|Servi...,340010.82,1.4
layerswap,203,203,45.74,LayerSwap fee,96390.97,4.7
glacis,19,9,28.49,Airlift Fee|Bridge Fee|Execution Gas|Network Gas|Service Fee,310253.12,0.9
symbiosis,10,10,11.47,CrossChain Fee,21642.08,5.3
stargateV2,8,8,4.93,LayerZero native fee,9140.68,5.4
polymer,55,24,4.91,CCTP forwarding fee|CCTP service fee|Polymer fee,14518.14,3.4
mayan,6,2,1.41,Execution Gas|Network Gas|Service Fee,1058878.60,0.0


## 11. Snapshots written by this notebook, the fetch log from the raw JSONLs, and the key check

In [25]:
# The fetch log, read from the raw JSONLs: calls, final HTTP statuses, span, observed rate, minimum ratelimit-remaining, largest gap between consecutive calls.
edge_recs = jsonl_read(EDGE_JSONL)
new_recs = [r for r in jsonl_read(S2600_JSONL) if r.get("sample_design") == "uniform_2000"]
fetch_log_table = pd.DataFrame({"edge cases (03 §4)": fetch_log(edge_recs), "uniform 2,000 (03 §6)": fetch_log(new_recs), "both fetches": fetch_log(edge_recs + new_recs)})
display(fetch_log_table)
print("The JSONL keeps each call's final response only (get_json retries HTTP 429 and 5xx before returning a record), so a retried 429 is not recorded. An HTTP 429 is returned only once the 100-per-minute window is exhausted, so the minimum ratelimit-remaining above (never below 41) is the evidence that none was returned. The largest gap between consecutive calls is descriptive: the pacing sleep plus the call's own latency.")
print("ratelimit-remaining distribution over both fetches:", pd.Series([int(r["ratelimit_remaining"]) for r in edge_recs + new_recs if str(r.get("ratelimit_remaining") or "").isdigit()]).describe(percentiles=[0.01, 0.1, 0.5]).round(1).to_dict())

,edge cases (03 §4),"uniform 2,000 (03 §6)",both fetches
calls,311,2000,2311
http_statuses,{200: 311},{200: 2000},{200: 2311}
first_call_utc,2026-10-28T19:01:00+00:00,2026-10-28T19:07:08+00:00,2026-10-28T19:01:00+00:00
last_call_utc,2026-10-28T19:07:07+00:00,2026-10-28T19:48:30+00:00,2026-10-28T19:48:30+00:00
calls_per_minute,50.8,48.3,48.7
min_ratelimit_remaining,41,45,41
max_gap_between_calls_s,4.0,5.0,5.0


The JSONL keeps each call's final response only (get_json retries HTTP 429 and 5xx before returning a record), so a retried 429 is not recorded. An HTTP 429 is returned only once the 100-per-minute window is exhausted, so the minimum ratelimit-remaining above (never below 41) is the evidence that none was returned. The largest gap between consecutive calls is descriptive: the pacing sleep plus the call's own latency.
ratelimit-remaining distribution over both fetches: {'count': 2311.0, 'mean': 74.7, 'std': 14.5, 'min': 41.0, '1%': 48.0, '10%': 55.0, '50%': 75.0, 'max': 99.0}


In [26]:
NEW_FILES = ["lifi_api_transfer_status_edge_cases.csv", "lifi_api_transfer_status_edge_cases_raw.jsonl", "lifi_api_transfer_status_sample_2600.csv", "lifi_api_transfer_status_sample_2600_raw.jsonl"]
m = manifest_read()
display(pd.DataFrame.from_dict({k: m[k] for k in NEW_FILES}, orient="index")[["fetched_at_utc", "rows", "notebook_section", "source_url"]])
for k in NEW_FILES:
    print(f"{k}: {m[k].get('note')}")
counts = {k: (len(pd.read_csv(REF / k, dtype=str)) if k.endswith(".csv") else sum(1 for _ in (REF / k).open())) for k in NEW_FILES}
print("rows on disk:", counts, " manifest agrees:", all(counts[k] == m[k]["rows"] for k in NEW_FILES))
if LIFI_KEY:
    leaked = [k for k in NEW_FILES if LIFI_KEY in (REF / k).read_text()]
    print("files under data/reference containing the key value:", leaked or "none")
    assert not leaked
print("files under data/reference:", sorted(p.name for p in REF.iterdir()))

,fetched_at_utc,rows,notebook_section,source_url
lifi_api_transfer_status_edge_cases.csv,2026-10-28T19:01:00+00:00,328,03 §4,"https://li.quest/v1/status?txHash=<tx_hash> (lookup tx_hash), http..."
lifi_api_transfer_status_edge_cases_raw.jsonl,2026-10-28T19:01:00+00:00,311,03 §4,"https://li.quest/v1/status?txHash=<tx_hash> (lookup tx_hash), http..."
lifi_api_transfer_status_sample_2600.csv,2026-10-28T19:07:08+00:00,2600,03 §6,https://li.quest/v1/status?txHash=<tx_hash>
lifi_api_transfer_status_sample_2600_raw.jsonl,2026-10-28T19:07:08+00:00,2600,03 §6,https://li.quest/v1/status?txHash=<tx_hash>


lifi_api_transfer_status_edge_cases.csv: 311 calls: {'tx_hash': 283, 'tx_hash_bridge': 22, 'transaction_id': 6}; 311 calls from 2026-10-28T19:01:00+00:00 to 2026-10-28T19:07:07+00:00 (51 per minute observed, paced at 0.65 s sleep after each call plus the call's own latency); final HTTP statuses {200: 311}; minimum ratelimit-remaining 41 of 100, so the window was never exhausted and no HTTP 429 was returned (a retried 429 would not be recorded, the JSONL keeping each call's final response); largest gap between consecutive calls 4 s; 294 edge rows over 11 reasons, random picks with DataFrame.sample(random_state=20261029); one CSV row per (edge row, lookup); raw responses in lifi_api_transfer_status_edge_cases_raw.jsonl
lifi_api_transfer_status_edge_cases_raw.jsonl: raw JSON responses, one line per call, with lookup kind, value, bridge parameter, request parameters (no key), HTTP status and ratelimit-remaining
lifi_api_transfer_status_sample_2600.csv: 2,000 events drawn uniformly from the